# 이전 adaptive 복귀 — 네 글자 선택 → 여러 생성 오류 → 실제 재역확산

**보정 코드는 안정화판 직전 d74e26e로 고정했습니다.** 새로운 거절/최종 선택/원본 대체 기능은 없습니다.
L4에서 1→2→3→4→5 실행. 3번에서 실제 글자 그림을 보고 네 칸을 선택합니다.
각 조합은 원래 조건인 seed 100 · strength .65로 한 장만 생성하고 그 이미지 한 장만 보정합니다.
정상 대조군과 다른 seed/강도를 자동으로 보정하지 않습니다. 다른 seed는 4번부터 새로 생성하세요.
여러 조합을 추가할 수도 있습니다. 모든 생성은 같은 기준 이미지에서 독립적으로 시작합니다.
이 실험은 선택한 기준 글자 4개를 img2img로 변형하는 통제 실험이며 프롬프트만으로 생성하는 시험이 아닙니다.

필요 자료: 기존 best.pt와 그 학습 폴더의 run_config.json/selected_paths.csv/conversion_report.json,
기준 글자·기준 획 원본 압축 자료(보통 target.zip/target_stroke.zip). Drive에서 기존 경로를 사용합니다.
전체 글자 목록은 기존 학습 목록에서 복구합니다. 네 글자만 든 디버그 ZIP으로 대체하지 않습니다.
체크포인트 SHA256을 확인합니다. 학습은 하지 않습니다.
5번 결과는 입력 / 재역확산만 / 이전 adaptive 보정입니다. 심한 실패도 자동으로 숨기지 않습니다.

In [ ]:
import sys,subprocess,json,hashlib,gc,zipfile,shutil
from pathlib import Path
from datetime import datetime
subprocess.run([sys.executable,'-m','pip','install','diffusers==0.37.0','transformers==4.57.6',
    'accelerate','safetensors','timm','numpy','pandas','tqdm','pillow','matplotlib',
    'scipy','scikit-image','ipywidgets'],check=True)
ADAPTIVE_COMMIT='d74e26e19d1e13360fdecde5d9d80d889344d990'
PINNED_HASHES={'diffusion_model/fitted_candidate_diffusion.py': '5fd3422b7b94a7443cbb639253b73e028744075ad3f55f2e7fab7511a77e8f0f', 'stroke_model/adaptive_ink_constraints.py': 'a3c10ce3c0cb4371f3fd72f6728afcb120f446e4bf8a31bd4e4272b42c321464', 'stroke_model/reference_model.py': 'df3445a74e2543ee9e86037b03316650918231f99da41dcf1efb0fb81b7f597f', 'stroke_model/stroke_reference_fit.py': '2c8cc24abcf3dfb555eff5622b63e5ab83460f802e39e36a706f38f2c96c89e1'}
RUNTIME_HELPERS={'diffusion_model/fitted_candidate_diffusion.py': '"""Paired DDIM re-denoising guided by an extractor-fitted candidate, not GT."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\n\n\ndef load_case(archive, case_id):\n    """Read named members only; recorded Colab absolute paths are not trusted."""\n    with zipfile.ZipFile(archive) as z:\n        if \'case_manifest.json\' not in z.namelist():\n            report = json.loads(z.read(\'report.json\'))\n            metadata = dict(report[\'metadata\'])\n            if metadata[\'case\'][\'case_id\'] != case_id:\n                raise ValueError(\'CASE_ID does not match the saved reverse experiment\')\n            with Image.open(io.BytesIO(z.read(\'input.png\'))) as im:\n                source = im.convert(\'RGB\').copy()\n            with Image.open(io.BytesIO(z.read(\'fitted_candidate.png\'))) as im:\n                target = im.convert(\'RGB\').copy()\n            if source.size != target.size or any(n % 8 for n in source.size):\n                raise ValueError(\'Input/candidate dimensions mismatch\')\n            metadata[\'reused_archive_kind\'] = \'reverse_result_original_input_and_candidate\'\n            return source, target, metadata\n        cases = json.loads(z.read(\'case_manifest.json\'))\n        matches = [c for c in cases if c[\'case_id\'] == case_id]\n        if len(matches) != 1:\n            raise ValueError(\'Select exactly one case from case_manifest.json\')\n        case = matches[0]\n        page = f"generation/page_{case[\'page_id\']:03d}.png"\n        candidate = f"{case_id}/normal_candidate.png"\n        def read_image(name):\n            with Image.open(io.BytesIO(z.read(name))) as im:\n                return im.convert(\'RGB\').copy()\n        source, target = read_image(page), read_image(candidate)\n        if source.size != target.size or any(n % 8 for n in source.size):\n            raise ValueError(\'Input and fitted candidate must have equal dimensions divisible by 8\')\n        report = json.loads(z.read(\'generation/generation_report.json\'))\n        metadata = dict(case=case, candidate_member=candidate, input_member=page,\n                        generation_model=report[\'model_id\'],\n                        candidate_is_ground_truth=False,\n                        prompt=report.get(\'prompts\', {}).get(\'4\', \'\'),\n                        negative_prompt=report.get(\'negative_prompt\', \'\'))\n    return source, target, metadata\n\n\ndef image_tensor(image, device):\n    a = np.asarray(image, dtype=np.float32) / 255\n    return torch.from_numpy(a.copy()).permute(2, 0, 1)[None].to(device)\n\n\nclass InkConstraintObjective:\n    """Candidate supplies fixed sparse constraints, never RGB target pixels."""\n    def __init__(self, source, candidate, *, enable_restore=False, tolerance=5.,\n                 halo=2, preserve_weight=.5, background_ink=.015, required_ink=.7):\n        from scipy.ndimage import distance_transform_edt, binary_dilation\n        from skimage.morphology import skeletonize\n        if source.shape != candidate.shape or source.shape[:2] != (1, 3):\n            raise ValueError(\'Expected matching single RGB sheets\')\n        if not np.isfinite(tolerance) or tolerance < 2 or type(halo) is not int or halo < 0:\n            raise ValueError(\'Invalid constraint tolerance/halo\')\n        if not 0 <= background_ink < required_ink <= 1 or preserve_weight < 0:\n            raise ValueError(\'Invalid ink constraints\')\n        self.source = source.detach()\n        self.preserve_weight = preserve_weight\n        self.background_ink, self.required_ink = background_ink, required_ink\n        source_ink = self.ink(source).detach().cpu().numpy()[0, 0]\n        target_ink = self.ink(candidate).detach().cpu().numpy()[0, 0]\n        source_binary, target_binary = source_ink > .35, target_ink > .35\n        if not source_binary.any() or not target_binary.any():\n            raise ValueError(\'Empty source or candidate: cannot infer safe correspondence\')\n        distance = distance_transform_edt(~target_binary)\n        # Ignore ordinary boundary/width differences within a tolerance corridor.\n        seeds = source_binary & (distance > tolerance)\n        nearby = binary_dilation(seeds, iterations=halo) if halo else seeds\n        erase = nearby*np.clip((distance-1)/max(tolerance-1, 1), 0, 1)\n        # Restore only an absent centerline, not the candidate\'s full silhouette.\n        missing = skeletonize(target_binary) & (distance_transform_edt(~source_binary) > tolerance)\n        restore = missing.astype(np.float32) if enable_restore else np.zeros_like(erase)\n        erase[restore > 0] = 0  # Never issue conflicting instructions.\n        def tensor(a):\n            return torch.as_tensor(a.copy(), dtype=torch.float32, device=source.device)[None, None]\n        self.erase, self.restore = tensor(erase), tensor(restore)\n        self.edit = torch.maximum(self.erase, self.restore)\n        occupied = (self.edit > 0).float()\n        self.preserve = 1-F.max_pool2d(occupied, 5, 1, 2)\n        self.metadata = dict(enable_restore=enable_restore, tolerance_pixels=tolerance,\n            halo_pixels=halo, preserve_weight=preserve_weight,\n            background_ink=background_ink, required_ink=required_ink,\n            erase_pixels=int((self.erase > 0).sum()), restore_pixels=int((self.restore > 0).sum()),\n            candidate_rgb_matching=False, masks_frozen_for_entire_pass=True)\n\n    @staticmethod\n    def ink(rgb):\n        return 1-(rgb*rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n\n    def project_gradient(self, gradient):\n        # A small latent context halo permits VAE reconstruction at edit boundaries.\n        support = F.adaptive_max_pool2d((self.edit > 0).float(), gradient.shape[-2:])\n        return gradient*F.max_pool2d(support, 3, 1, 1)\n\n    def __call__(self, image):\n        ink = self.ink(image)\n        remaining = F.relu(ink-self.background_ink)\n        erase = CandidateObjective.mean_region(remaining, self.erase)\n        # Penalize the darkest residual pixels as well as average remaining ink.\n        active = remaining[self.erase > 0]\n        tail = active.topk(max(1, int(active.numel()*.1))).values.mean() if active.numel() else erase*0\n        restore = CandidateObjective.mean_region(F.relu(self.required_ink-ink), self.restore)\n        preserve = CandidateObjective.mean_region((image-self.source).square(), self.preserve)\n        healthy_ink = self.preserve*(self.ink(self.source) > .2).float()\n        healthy_mse = CandidateObjective.mean_region((image-self.source).square(), healthy_ink)\n        loss = erase+.25*tail+restore+self.preserve_weight*preserve\n        return loss, dict(erase_ink=float(erase.detach()), erase_tail=float(tail.detach()),\n            restore_deficit=float(restore.detach()), preserved_region_mse=float(preserve.detach()),\n            healthy_ink_rgb_mse=float(healthy_mse.detach()))\n\n\nclass CandidateObjective:\n    """Fixed spatial weights; gradients stay continuous through the VAE."""\n    def __init__(self, source, candidate, threshold=.10, radius=2, preserve_weight=.25):\n        if source.shape != candidate.shape or not 0 < threshold < 1 or radius < 0:\n            raise ValueError(\'Invalid images or edit-region settings\')\n        self.source, self.candidate = source.detach(), candidate.detach()\n        difference = (source - candidate).abs().mean(1, keepdim=True)\n        edit = (difference > threshold).float()\n        self.edit = F.max_pool2d(edit, 2*radius+1, 1, radius).detach()\n        self.preserve_weight = preserve_weight\n\n    @staticmethod\n    def mean_region(values, mask):\n        return (values*mask).sum() / (mask.sum()*values.shape[1]).clamp_min(1)\n\n    def __call__(self, image):\n        match = self.mean_region((image-self.candidate).square(), self.edit)\n        preserve = self.mean_region((image-self.source).square(), 1-self.edit)\n        loss = match + self.preserve_weight*preserve\n        return loss, dict(candidate_region_mse=float(match.detach()),\n                          preserved_region_mse=float(preserve.detach()))\n\n\ndef optimize_clean(clean, decode, objective, *, iterations=4, step_ratio=.02,\n                   budget_ratio=.08, backtracks=5):\n    """Backtrack on the SAME clean estimate; bound total correction per step."""\n    anchor = clean.detach().float()\n    current = anchor.clone()\n    scale = anchor.square().mean().sqrt().clamp_min(1e-3)\n    with torch.no_grad():\n        before, _ = objective(decode(current))\n    accepted = 0\n    for _ in range(iterations):\n        with torch.enable_grad():\n            leaf = current.detach().requires_grad_(True)\n            loss, _ = objective(decode(leaf))\n            gradient, = torch.autograd.grad(loss, leaf)\n            if hasattr(objective, \'project_gradient\'):\n                gradient = objective.project_gradient(gradient)\n        if not torch.isfinite(gradient).all() or not torch.isfinite(loss):\n            raise FloatingPointError(\'Nonfinite candidate guidance\')\n        norm = gradient.square().mean().sqrt()\n        if norm < 1e-12:\n            break\n        direction = step_ratio*scale*gradient/norm\n        found = False\n        with torch.no_grad():\n            for attempt in range(backtracks+1):\n                proposal = current-direction*(.5**attempt)\n                delta = proposal-anchor\n                rms = delta.square().mean().sqrt().clamp_min(1e-12)\n                proposal = anchor+delta*(budget_ratio*scale/rms).clamp(max=1)\n                trial, _ = objective(decode(proposal))\n                if torch.isfinite(trial) and trial < loss.detach():\n                    current = proposal.detach()\n                    accepted += 1\n                    found = True\n                    break\n        if not found:\n            break\n    with torch.no_grad():\n        after, details = objective(decode(current))\n    return current, dict(loss_before=float(before), loss_after=float(after),\n                         accepted_inner_steps=accepted, **details)\n\n\ndef run_pair(source, candidate, metadata, output, *, steps=60, strength=.35,\n             seed=2026, cfg_scale=5., inner_steps=4, step_ratio=.02,\n             model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\', revision=None,\n             progress=None, guidance_mode=\'candidate\', constraint_config=None, objective_override=None):\n    """Identical source VAE encoding, noise, prompt, scheduler in both arms."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select an L4 GPU runtime in Colab\')\n    if not 0 < strength <= 1 or steps < 4 or int(steps*strength) < 2:\n        raise ValueError(\'Need at least two actual re-denoising steps\')\n    if inner_steps < 1 or step_ratio <= 0 or cfg_scale <= 1:\n        raise ValueError(\'Invalid guidance configuration\')\n    if guidance_mode not in (\'candidate\', \'ink_constraints\', \'adaptive_ink\'):\n        raise ValueError(\'Unknown guidance mode\')\n    if (guidance_mode == \'adaptive_ink\') != (objective_override is not None):\n        raise ValueError(\'Adaptive mode requires an explicitly prepared stroke objective\')\n    from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n    from copy import deepcopy\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    source.save(out/\'input.png\')\n    candidate.save(out/\'fitted_candidate.png\')\n    kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n    if revision:\n        kwargs[\'revision\'] = revision\n    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config,\n                                              clip_sample=False, thresholding=False)\n    pipe.to(\'cuda\')\n    pipe.enable_attention_slicing()\n    pipe.enable_vae_slicing()\n    for module in (pipe.unet, pipe.vae, pipe.text_encoder):\n        module.eval().requires_grad_(False)\n    prompt = metadata.get(\'prompt\') or (\'four Korean Hangul glyphs, black ink on white, \'\n                                      \'preserve the input typography and layout\')\n    with torch.no_grad():\n        positive, negative = pipe.encode_prompt(prompt, \'cuda\', 1, True,\n            negative_prompt=metadata.get(\'negative_prompt\', \'\'))\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        # VAE and noise initialization exactly once, then copy for both arms.\n        initial = pipe.prepare_latents(pipe.image_processor.preprocess(source), timesteps[:1],\n            1, 1, embeddings.dtype, \'cuda\', torch.Generator(\'cpu\').manual_seed(seed))\n    pipe.vae.to(dtype=torch.float32)\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor,\n                               return_dict=False)[0].float()/2+.5\n    def save_image(z, path):\n        with torch.no_grad():\n            array = decode(z)[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(array*255).astype(\'uint8\')).save(path)\n    source_rgb, candidate_rgb = image_tensor(source, \'cuda\'), image_tensor(candidate, \'cuda\')\n    if objective_override is not None and (objective_override.source.shape != source_rgb.shape or\n            not torch.allclose(objective_override.source, source_rgb, atol=1e-6)):\n        raise ValueError(\'Adaptive objective must be prepared from this exact original input\')\n    objective = objective_override if objective_override is not None else (InkConstraintObjective(source_rgb, candidate_rgb, **(constraint_config or {}))\n                 if guidance_mode == \'ink_constraints\' else CandidateObjective(source_rgb, candidate_rgb))\n    Image.fromarray((objective.edit[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/\'edit_region.png\')\n    if guidance_mode in (\'ink_constraints\', \'adaptive_ink\'):\n        for name in (\'erase\', \'restore\', \'preserve\'):\n            Image.fromarray(np.rint(getattr(objective, name)[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/f\'{name}_region.png\')\n        (out/\'constraints.json\').write_text(json.dumps(objective.metadata, indent=2), encoding=\'utf-8\')\n    guided_name = {\'candidate\': \'candidate_guided\', \'ink_constraints\': \'ink_guided\', \'adaptive_ink\': \'adaptive_guided\'}[guidance_mode]\n    if hasattr(objective, \'export_state\'):\n        objective.export_state(out/\'ownership_initial.npz\')\n    traces, metrics = {}, {}\n    for method in (\'redenoise_only\', guided_name):\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.detach().float().clone()\n        trace = []\n        for i, t in enumerate(timesteps):\n            with torch.no_grad():\n                inputs = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(inputs.to(torch.float16), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                noise = (u+cfg_scale*(c-u)).float()\n                base = scheduler.step(noise, t, latent, eta=0., return_dict=True)\n            record = dict(step=i, timestep=int(t), active=False, accepted_inner_steps=0)\n            latent = base.prev_sample.detach()\n            clean = base.pred_original_sample.detach()\n            if method == guided_name and i/max(len(timesteps)-1, 1) >= .2:\n                if hasattr(objective, \'maybe_refresh\'):\n                    with torch.no_grad():\n                        refresh = objective.maybe_refresh(decode(clean), i)\n                    if refresh is not None:\n                        record[\'constraint_refresh\'] = refresh\n                        for name in (\'erase\', \'restore\'):\n                            mask = getattr(objective, name)[0, 0].detach().cpu().numpy()\n                            Image.fromarray(np.rint(mask*255).clip(0, 255).astype(\'uint8\')).save(out/f\'{name}_step_{i:03d}.png\')\n                        if hasattr(objective, \'export_state\'):\n                            objective.export_state(out/f\'ownership_step_{i:03d}.npz\')\n                clean, result = optimize_clean(clean, decode, objective,\n                    iterations=inner_steps if i < len(timesteps)-1 else inner_steps*2,\n                    step_ratio=step_ratio)\n                # DDIM eta=0: x_(t-1) = sqrt(a_prev)*x0 + sqrt(1-a_prev)*epsilon.\n                # Hold the predicted epsilon fixed and change the clean component.\n                prev_t = int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[prev_t] if prev_t >= 0\n                         else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent+alpha.sqrt()*(clean-base.pred_original_sample)\n                record.update(active=True, **result)\n            latent = latent.detach()\n            trace.append(record)\n            if i % 4 == 0 or i == len(timesteps)-1:\n                save_image(clean, out/f\'{method}_step_{i:03d}.png\')\n            if progress:\n                progress(method, i+1, len(timesteps), record)\n        save_image(latent, out/f\'{method}.png\')\n        with torch.no_grad():\n            evaluate = getattr(objective, \'evaluate_fixed\', objective)\n            loss, values = evaluate(decode(latent))\n        metrics[method] = dict(loss=float(loss), **values)\n        traces[method] = trace\n    with torch.no_grad():\n        loss, values = getattr(objective, \'evaluate_fixed\', objective)(objective.source)\n    metrics[\'input\'] = dict(loss=float(loss), **values)\n    report = dict(metadata=metadata, model_id=model_id, revision=revision,\n        steps=steps, actual_steps=len(timesteps), strength=strength, seed=seed,\n        cfg_scale=cfg_scale, inner_steps=inner_steps, step_ratio=step_ratio,\n        initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n        paired_initial_latent=True, guidance_target=guidance_mode,\n        constraints=getattr(objective, \'metadata\', None),\n        constraint_history=getattr(objective, \'history\', None),\n        target_pixels_pasted_into_output=False, metrics=metrics, traces=traces,\n        metric_note=(\'Fixed ink constraints and source preservation; not GT accuracy or HSQM score\'\n                     if guidance_mode in (\'ink_constraints\', \'adaptive_ink\') else \'Candidate agreement, not ground-truth accuracy or HSQM score\'))\n    (out/\'report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    del pipe\n    torch.cuda.empty_cache()\n    return report\n', 'stroke_model/adaptive_ink_constraints.py': '"""Frozen stroke identities, per-stroke bands, periodically refreshed ownership.\n\nGeometry is anchored to the initial fitted strokes, not re-fitted to new errors.\nExtractor softmax is detached; only image ink losses backpropagate through VAE.\n"""\nimport hashlib\nimport io\nimport json\nimport zipfile\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom scipy.ndimage import distance_transform_edt, binary_dilation\nfrom skimage.morphology import skeletonize\n\nfrom diffusion_model.fitted_candidate_diffusion import InkConstraintObjective, CandidateObjective\nfrom .stroke_reference_fit import _component, principal_path\n\n\ndef checkpoint_hash(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef load_stroke_bundle(archive, case_id):\n    """Only known ZIP members are read, no recorded absolute paths are followed."""\n    with zipfile.ZipFile(archive) as z:\n        info = json.loads(z.read(\'experiment_info.json\'))\n        manifest = json.loads(z.read(\'source_conditions/conditions.json\'))\n        records = []\n        if len(manifest) != 4:\n            raise ValueError(\'Exactly four reference slots required\')\n        for i, entry in enumerate(manifest):\n            name = entry[\'path\'].replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n            with np.load(io.BytesIO(z.read(\'source_conditions/\'+name)), allow_pickle=False) as d:\n                reference, strokes = d[\'reference\'].copy(), d[\'strokes\'].copy()\n            with np.load(io.BytesIO(z.read(f\'{case_id}/warped_slot_{i+1}.npz\')), allow_pickle=False) as d:\n                fitted = d[\'warped_strokes\'].copy()\n            if strokes.shape != fitted.shape or reference.shape != (1, *strokes.shape[-2:]):\n                raise ValueError(\'Reference/fitted stroke shape mismatch\')\n            records.append(dict(reference=reference, strokes=strokes, fitted=fitted))\n        return records, info[\'checkpoint_sha256\']\n\n\nclass AdaptiveInkConstraints(InkConstraintObjective):\n    def __init__(self, source, extractor, records, *, refresh_every=3, enable_restore=False,\n                 margin_ratio=.2, margin_min=.6, margin_max=1.8,\n                 ownership_blend=.35, mask_blend=.5, preserve_weight=.5, continuity_weight=.2):\n        if source.shape[:2] != (1, 3) or len(records) != 4:\n            raise ValueError(\'One RGB sheet and four stroke records required\')\n        if type(refresh_every) is not int or refresh_every < 1 or not 0 < ownership_blend <= 1 or not 0 < mask_blend <= 1:\n            raise ValueError(\'Invalid refresh schedule/blending\')\n        if not 0 <= margin_ratio or not 0 < margin_min <= margin_max or preserve_weight < 0 or continuity_weight < 0:\n            raise ValueError(\'Invalid stroke bands/weights\')\n        self.source = source.detach()\n        self.extractor = extractor.eval().requires_grad_(False)\n        self.refresh_every, self.ownership_blend, self.mask_blend = refresh_every, ownership_blend, mask_blend\n        self.enable_restore = enable_restore\n        self.preserve_weight, self.continuity_weight = preserve_weight, continuity_weight\n        self.background_ink, self.required_ink = .015, .7\n        self.records, self.bands, self.history = [], [], []\n        h, w = source.shape[-2:]\n        if h % 2 or w % 2:\n            raise ValueError(\'Equal 2x2 slots require even sheet dimensions\')\n        boxes = [(0, 0, w//2, h//2), (w//2, 0, w, h//2),\n                 (0, h//2, w//2, h), (w//2, h//2, w, h)]\n        source_ink = self.ink(source)\n        # Identity k always refers to input canonical stroke k, never sorted/merged.\n        for slot, (raw, box) in enumerate(zip(records, boxes)):\n            ref = torch.as_tensor(raw[\'reference\'], device=source.device, dtype=torch.float32)[None]\n            strokes = torch.as_tensor(raw[\'strokes\'], device=source.device, dtype=torch.float32)[None]\n            fitted = np.asarray(raw[\'fitted\'], np.float32)\n            crop = self.crop(source_ink, box, fitted.shape[-2:])\n            probabilities = self.predict(crop, ref, strokes)\n            observed = crop[0, 0].cpu().numpy()\n            distances, radii, margins, cores, source_bodies = [], [], [], [], []\n            for k, mask in enumerate(fitted):\n                binary = mask > .35\n                skeleton = skeletonize(binary)\n                if not skeleton.any():\n                    raise ValueError(f\'Slot {slot+1}, stroke {k+1}: empty fitted path\')\n                inside = distance_transform_edt(binary)\n                distance, nearest = distance_transform_edt(~skeleton, return_indices=True)\n                expected = max(float(np.median(inside[skeleton])), .75)\n                body = (observed > .35) & (probabilities[k] > .35)\n                width_samples = distance_transform_edt(body)[skeleton & body]\n                # Only samples on the anchored main path calibrate width. Extras\n                # off the path cannot enlarge the allowed corridor arbitrarily.\n                observed_half = float(np.median(width_samples)) if width_samples.size >= 4 else expected\n                half = float(np.clip(observed_half, .75*expected, 1.25*expected))\n                local_radius = np.maximum(inside[tuple(nearest)], .75)*(half/expected)\n                margin = float(np.clip(margin_ratio*2*half, margin_min, margin_max))\n                trusted_core = skeleton & (observed > .35)\n                source_body = np.zeros_like(binary)\n                path_protected = False\n                try:\n                    component, _ = _component(probabilities[k]*observed, hint=mask)\n                    path, path_info = principal_path(component, n=128)\n                    length = float(np.linalg.norm(np.diff(path, axis=0), axis=1).sum())\n                    ratio = length/max(int(skeleton.sum()), 1)\n                    if not path_info[\'closed\'] and .65 <= ratio <= 1.5 and path_info[\'path_coverage\'] > .65:\n                        dense = np.stack([np.interp(np.linspace(0, 1, 512), np.linspace(0, 1, len(path)), path[:, j]) for j in range(2)], 1)\n                        xy = np.rint(dense).astype(int)\n                        xy[:, 0] = xy[:, 0].clip(0, binary.shape[1]-1)\n                        xy[:, 1] = xy[:, 1].clip(0, binary.shape[0]-1)\n                        source_path = np.zeros_like(binary);source_path[xy[:, 1], xy[:, 0]] = True\n                        if float(np.median(distance[source_path])) <= expected+margin:\n                            # Protect the original main body, not the smooth\n                            # candidate outline. Cap radius to exclude side branches.\n                            source_body = (distance_transform_edt(~source_path) <= 1.1*half) & (observed > .35)\n                            trusted_core = source_path & (observed > .35)\n                            path_protected = True\n                except ValueError:\n                    pass  # Explicitly reported fallback to anchored-path protection.\n                distances.append(distance); radii.append(local_radius); margins.append(margin); cores.append(trusted_core)\n                source_bodies.append(source_body)\n                self.bands.append(dict(slot=slot+1, stroke=k+1, half_width_model=half,\n                    tolerance_model=margin, tolerance_sheet_x=margin*(box[2]-box[0])/fitted.shape[-1],\n                    tolerance_sheet_y=margin*(box[3]-box[1])/fitted.shape[-2],\n                    trusted_core_pixels=int(trusted_core.sum()), original_main_path_protected=path_protected))\n            self.records.append(dict(reference=ref, strokes=strokes, box=box,\n                size=fitted.shape[-2:], probabilities=probabilities,\n                distances=np.stack(distances), radii=np.stack(radii), margins=np.asarray(margins)[:, None, None],\n                cores=np.stack(cores), original_bodies=np.stack(source_bodies),\n                skeletons=np.stack([skeletonize(s > .35) for s in fitted])))\n        self.erase = torch.zeros_like(source_ink)\n        self.restore = torch.zeros_like(source_ink)\n        self.protected_cores = []\n        for record in self.records:\n            for core in record[\'cores\']:\n                canvas = torch.zeros_like(source_ink)\n                self.place(canvas, self.tensor(core.astype(np.float32)), record[\'box\'])\n                self.protected_cores.append(canvas)\n        self.refresh(source, step=-1, initial=True)\n        self.fixed = dict(erase=self.erase.clone(), restore=self.restore.clone(), preserve=self.preserve.clone())\n        self.metadata = dict(kind=\'adaptive_stroke_ink_constraints\', candidate_rgb_matching=False,\n            enable_restore=enable_restore, refresh_every=refresh_every, stroke_identities_frozen=True,\n            geometry_frozen=True, width_calibration=\'robust trusted main-path samples, frozen\',\n            ownership_backward=False, bands=self.bands, preserve_weight=preserve_weight,\n            continuity_weight=continuity_weight, masks_frozen_for_entire_pass=False)\n\n    def tensor(self, a):\n        return torch.as_tensor(np.asarray(a).copy(), dtype=torch.float32, device=self.source.device)[None, None]\n\n    @staticmethod\n    def crop(image, box, size):\n        x0, y0, x1, y1 = box\n        return F.interpolate(image[..., y0:y1, x0:x1], size=size, mode=\'bilinear\', align_corners=False)\n\n    @staticmethod\n    def place(canvas, patch, box):\n        x0, y0, x1, y1 = box\n        canvas[..., y0:y1, x0:x1] = F.interpolate(patch, size=(y1-y0, x1-x0), mode=\'bilinear\', align_corners=False)\n\n    @torch.no_grad()\n    def predict(self, ink, reference, strokes):\n        p = self.extractor(ink, reference, strokes)[\'probabilities\'][0].float()\n        if p.shape != strokes.shape[1:] or not torch.isfinite(p).all() or (p < 0).any():\n            raise ValueError(\'Invalid extractor ownership probabilities\')\n        p = p/p.sum(0, keepdim=True).clamp_min(1e-8)\n        return p.cpu().numpy()\n\n    @torch.no_grad()\n    def refresh(self, image, *, step, initial=False):\n        ink = self.ink(image).clamp(0, 1)\n        erase, restore = torch.zeros_like(ink), torch.zeros_like(ink)\n        rows = []\n        for slot, r in enumerate(self.records):\n            crop = self.crop(ink, r[\'box\'], r[\'size\'])\n            current = self.predict(crop, r[\'reference\'], r[\'strokes\'])\n            p = current if initial else (1-self.ownership_blend)*r[\'probabilities\']+self.ownership_blend*current\n            r[\'probabilities\'] = p\n            observed = crop[0, 0].cpu().numpy()\n            excess = r[\'distances\']-r[\'radii\']\n            outside = np.clip((excess-r[\'margins\'])/np.maximum(r[\'margins\'], .1), 0, 1)\n            # Protect every other legitimate stroke at intersections too.\n            safe_body = (excess <= r[\'margins\']).any(0) | r[\'original_bodies\'].any(0)\n            responsibility = (p*outside).sum(0)\n            # Existing faint residuals keep their old mask. New regions require\n            # more than faint decoder noise before they can expand deletion.\n            seeds = (observed > (.25 if initial else .12)) & (responsibility > .4) & ~safe_body\n            grown = binary_dilation(seeds, iterations=1)\n            weight = grown*responsibility*(~safe_body)\n            missing = r[\'skeletons\'].any(0) & (distance_transform_edt(~(observed > .2)) > 1.)\n            restore_weight = missing.astype(np.float32) if self.enable_restore else np.zeros_like(weight)\n            weight[restore_weight > 0] = 0\n            self.place(erase, self.tensor(weight), r[\'box\'])\n            self.place(restore, self.tensor(restore_weight), r[\'box\'])\n            for k in range(len(p)):\n                rows.append(dict(slot=slot+1, stroke=k+1,\n                    assigned_ink=float((p[k]*observed).sum()),\n                    outside_ink=float((p[k]*observed*outside[k]*(~safe_body)).sum())))\n        # Retain previous deletion support: fading errors must not fall out of\n        # the mask or become protected just because they were partly erased.\n        self.erase = erase if initial else torch.maximum(self.erase, self.mask_blend*erase)\n        self.restore = restore if initial else (1-self.mask_blend)*self.restore+self.mask_blend*restore\n        self.erase = self.erase*(self.restore <= 0).float()\n        self.edit = torch.maximum(self.erase, self.restore)\n        self.preserve = 1-F.max_pool2d((self.edit > .01).float(), 5, 1, 2)\n        row = dict(step=step, erase_pixels=int((self.erase > .01).sum()),\n                   restore_pixels=int((self.restore > .01).sum()), per_stroke=rows)\n        self.history.append(row)\n        return row\n\n    def maybe_refresh(self, image, step):\n        return self.refresh(image, step=step) if step % self.refresh_every == 0 else None\n\n    def export_state(self, path):\n        arrays = {f\'slot_{i+1}_probabilities\': r[\'probabilities\'] for i, r in enumerate(self.records)}\n        arrays.update(erase=self.erase.detach().cpu().numpy(), restore=self.restore.detach().cpu().numpy())\n        np.savez_compressed(path, **arrays)\n\n    def __call__(self, image):\n        loss, details = super().__call__(image)\n        ink = self.ink(image)\n        # Protect initially present main paths, averaged per stroke, not per sheet.\n        continuity = [CandidateObjective.mean_region(F.relu(.65-ink), core)\n                      for core in self.protected_cores if float(core.sum()) > 0]\n        value = torch.stack(continuity).mean() if continuity else loss*0\n        details[\'trusted_path_deficit\'] = float(value.detach())\n        return loss+self.continuity_weight*value, details\n\n    def evaluate_fixed(self, image):\n        current = dict(erase=self.erase, restore=self.restore, preserve=self.preserve)\n        try:\n            for key, value in self.fixed.items():\n                setattr(self, key, value)\n            return self(image)\n        finally:\n            for key, value in current.items():\n                setattr(self, key, value)\n', 'stroke_model/reference_model.py': '"""Reference-conditioned V2. Inputs: white ink on black, Bx1xHxW in [0,1].\n\nNo thresholding, CPU conversion or detached image features in the forward path.\nOld checkpoints are deliberately not compatible with this new architecture.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef block(cin, cout, stride=1):\n    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride=stride, padding=1),\n                         nn.GroupNorm(4, cout), nn.GELU())\n\n\ndef resize(x, size):\n    return F.interpolate(x, size=size, mode="bilinear", align_corners=False)\n\n\ndef warp(strokes, flow):\n    b, _, h, w = strokes.shape\n    ys = (torch.arange(h, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / h - 1\n    xs = (torch.arange(w, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / w - 1\n    y, x = torch.meshgrid(ys, xs, indexing="ij")\n    grid = torch.stack((x, y), -1)[None].expand(b, -1, -1, -1)\n    offset = torch.stack((flow[:, 0] * 2 / w, flow[:, 1] * 2 / h), -1)\n    return F.grid_sample(strokes, grid + offset, align_corners=False)\n\n\nclass ReferenceStrokeModel(nn.Module):\n    def __init__(self, image_size=128, width=64, pretrained=True,\n                 backbone=None, channels=None, channels_last=False, stroke_chunk_size=4):\n        super().__init__()\n        if width % 4:\n            raise ValueError("width must be divisible by 4")\n        mean, std = (.485, .456, .406), (.229, .224, .225)\n        if backbone is None:\n            import timm\n            backbone = timm.create_model(\n                "swin_tiny_patch4_window7_224", pretrained=pretrained,\n                features_only=True, out_indices=(0, 1, 2, 3),\n                in_chans=3, img_size=image_size, drop_path_rate=0.)\n            channels = backbone.feature_info.channels()\n            channels_last = True  # timm Swin features are NHWC.\n            mean = backbone.pretrained_cfg.get("mean", mean)\n            std = backbone.pretrained_cfg.get("std", std)\n        if not channels:\n            raise ValueError("Injected backbone requires its feature channels")\n        self.backbone = backbone\n        self.stroke_chunk_size = max(1, int(stroke_chunk_size))\n        self.channels_last = channels_last\n        self.backbone_frozen = True\n        self.backbone.requires_grad_(False)\n        self.backbone.eval()\n        self.register_buffer("rgb_mean", torch.tensor(mean).view(1, 3, 1, 1))\n        self.register_buffer("rgb_std", torch.tensor(std).view(1, 3, 1, 1))\n        self.lateral = nn.ModuleList(nn.Conv2d(c, width, 1) for c in channels)\n        self.smooth = nn.ModuleList(block(width, width) for _ in channels)\n        self.detail = block(1, width)  # Full-resolution branch for thin strokes.\n        self.fuse = block(width * 2, width)\n        self.flow_net = nn.Sequential(block(2, 16, 2), block(16, 32, 2),\n                                      nn.Conv2d(32, 2, 3, padding=1))\n        nn.init.zeros_(self.flow_net[-1].weight)\n        nn.init.zeros_(self.flow_net[-1].bias)\n        # Shared across stroke IDs; full glyph + selected stroke + xy context.\n        self.reference = nn.Sequential(block(4, 16, 2), block(16, width, 2))\n        self.head = nn.Sequential(block(width * 2 + 4, width),\n                                  nn.Conv2d(width, 1, 1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        if self.backbone_frozen:\n            self.backbone.eval()\n        return self\n\n    def forward(self, image, reference, strokes, active=None):\n        if image.ndim != 4 or image.shape[1] != 1 or reference.shape != image.shape:\n            raise ValueError("image/reference must have matching Bx1xHxW shapes")\n        b, _, h, w = image.shape\n        if strokes.ndim != 4 or strokes.shape[0] != b or strokes.shape[-2:] != (h, w):\n            raise ValueError("strokes must have shape BxKxHxW")\n        k = strokes.shape[1]\n        if active is None:\n            active = strokes.flatten(2).sum(-1) > 0  # Fixed reference metadata only.\n        if active.shape != (b, k) or active.dtype != torch.bool or not active.any(1).all():\n            raise ValueError("Each sample needs at least one active reference stroke")\n        rgb = (1 - image).expand(-1, 3, -1, -1)\n        feats = self.backbone((rgb - self.rgb_mean) / self.rgb_std)\n        if self.channels_last:\n            feats = [f.permute(0, 3, 1, 2) for f in feats]\n        pyramid = None\n        for i in reversed(range(len(feats))):\n            lateral = self.lateral[i](feats[i])\n            pyramid = lateral if pyramid is None else lateral + resize(pyramid, lateral.shape[-2:])\n            pyramid = self.smooth[i](pyramid)\n        features = self.fuse(torch.cat((resize(pyramid, (h, w)), self.detail(image)), 1))\n        flow = 12 * resize(self.flow_net(torch.cat((reference, image), 1)), (h, w)).tanh()\n        hints = warp(strokes, flow)\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=image.device, dtype=image.dtype),\n                                torch.linspace(-1, 1, w, device=image.device, dtype=image.dtype), indexing="ij")\n        xy = torch.stack((xx, yy))[None].expand(b, -1, -1, -1)\n        logits = self.score_strokes(features, reference, strokes, xy, flow, hints, image)\n        logits = logits.float().masked_fill(~active[:, :, None, None], -torch.inf)\n        probabilities = logits.softmax(1) * active[:, :, None, None]\n        probabilities = probabilities / probabilities.sum(1, keepdim=True)\n        return {"logits": logits, "probabilities": probabilities,\n                "masks": image * probabilities, "flow": flow, "hints": hints,\n                "active": active}\n\n    def score_strokes(self, features, reference, strokes, xy, flow, hints, image):\n        """Same head algebra, batching strokes and computing shared convolution once.\n\n        GroupNorm is per sample, so stroke batching does not mix stroke statistics.\n        Existing V2 state_dicts remain compatible; input gradients remain connected.\n        """\n        b, k, h, w = strokes.shape\n        width = features.shape[1]\n        conv = self.head[0][0]\n        shared_weight = torch.cat((conv.weight[:, :width], conv.weight[:, 2*width+1:]), 1)\n        shared = F.conv2d(torch.cat((features, image, xy), 1), shared_weight, conv.bias, padding=1)\n        outputs = []\n        for start in range(0, k, self.stroke_chunk_size):\n            n = min(self.stroke_chunk_size, k-start)\n            def repeat(t):\n                return t[:, None].expand(-1, n, -1, -1, -1).reshape(b*n, *t.shape[1:])\n            selected = strokes[:, start:start+n].reshape(b*n, 1, h, w)\n            r = self.reference(torch.cat((repeat(reference), selected, repeat(xy)), 1))\n            r = warp(resize(r, (h, w)), repeat(flow))\n            condition = torch.cat((r, hints[:, start:start+n].reshape(b*n, 1, h, w)), 1)\n            x = F.conv2d(condition, conv.weight[:, width:2*width+1], padding=1) + repeat(shared)\n            x = self.head[0][2](self.head[0][1](x))\n            outputs.append(self.head[1](x).reshape(b, n, h, w))\n        return torch.cat(outputs, 1)\n\n\ndef exclusive_masks(output, foreground):\n    """Export only: non-differentiable exact partition of the supplied foreground."""\n    expected = output[\'probabilities\'][:, :1].shape\n    if foreground.shape != expected:\n        raise ValueError(\'foreground must have shape Bx1xHxW\')\n    labels = output["probabilities"].argmax(1)\n    masks = F.one_hot(labels, output["probabilities"].shape[1]).permute(0, 3, 1, 2)\n    return masks.bool() & foreground.bool()\n\n\ndef stroke_moments(masks, eps=1e-6):\n    """Area fraction, centroid x/y and xx/xy/yy covariance in normalized coordinates.\n\n    Near-empty strokes have unreliable geometry: callers must mask/inspect them.\n    """\n    h, w = masks.shape[-2:]\n    y, x = torch.meshgrid(torch.linspace(-1, 1, h, device=masks.device, dtype=masks.dtype),\n                          torch.linspace(-1, 1, w, device=masks.device, dtype=masks.dtype), indexing="ij")\n    mass = masks.sum((-2, -1))\n    weights = masks / mass.clamp_min(eps)[..., None, None]\n    cx, cy = (weights * x).sum((-2, -1)), (weights * y).sum((-2, -1))\n    dx, dy = x - cx[..., None, None], y - cy[..., None, None]\n    return torch.stack((mass / (h * w), cx, cy,\n                        (weights * dx.square()).sum((-2, -1)),\n                        (weights * dx * dy).sum((-2, -1)),\n                        (weights * dy.square()).sum((-2, -1))), -1)\n\n\ndef partition_loss(output, labels):\n    """labels: BxHxW, 0..K-1 for owned pixels; -1 background/unknown.\n\n    Unknown crossing pixels are excluded, not assigned pseudo-certainty.\n    Soft Dice averages only annotated, nonempty target strokes.\n    """\n    logits = output["logits"]\n    valid = labels >= 0\n    if labels.shape != logits.shape[:1] + logits.shape[2:]:\n        raise ValueError("labels must have shape BxHxW")\n    if not valid.any() or (labels < -1).any() or (labels >= logits.shape[1]).any():\n        raise ValueError("Need annotated pixels with labels in -1..K-1")\n    target = F.one_hot(labels.clamp_min(0).long(), logits.shape[1]).permute(0, 3, 1, 2)\n    target = target * valid[:, None]\n    if (target.bool() & ~output["active"][:, :, None, None]).any():\n        raise ValueError("Target refers to an inactive reference stroke")\n    ce = F.cross_entropy(logits, labels.long(), ignore_index=-1)\n    pred = output["probabilities"] * valid[:, None]\n    mass = target.sum((-2, -1))\n    dice = (2 * (pred * target).sum((-2, -1)) + 1e-6) / (pred.sum((-2, -1)) + mass + 1e-6)\n    dice_loss = (1 - dice)[mass > 0].mean()\n    flow = output["flow"]\n    smooth = (flow[..., 1:, :] - flow[..., :-1, :]).square().mean()\n    smooth = smooth + (flow[..., :, 1:] - flow[..., :, :-1]).square().mean()\n    return {"loss": ce + dice_loss + .001 * smooth, "ce": ce,\n            "dice_loss": dice_loss, "flow_smoothness": smooth}\n', 'stroke_model/stroke_reference_fit.py': '"""Fit valid reference strokes to extracted stroke positions, not their defects.\n\nInference-only geometry: principal skeleton paths discard short side branches;\nstraight references remain straight, curves receive a regularized deformation.\nThe original reference centerline is deformed and re-rendered at the observed\nwidth. Output is a candidate for inspection, not a ground-truth reconstruction.\n"""\nfrom dataclasses import dataclass, asdict\nimport heapq\nimport numpy as np\nfrom scipy.ndimage import label, distance_transform_edt, map_coordinates, affine_transform,gaussian_filter1d\nfrom scipy.spatial import cKDTree\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass ReferenceFitConfig:\n    points: int=64\n    smoothness: float=12.\n    shape_prior: float=.15\n    max_displacement: float=16.\n    max_width: float=10.\n    min_width: float=1.\n    min_length_ratio: float=.65\n    max_length_ratio: float=1.6\n    junction_radius: float=2.5\n    junction_weight: float=8.\n    max_junction_shift: float=2.\n\n    def __post_init__(self):\n        if type(self.points) is not int or self.points<8:raise ValueError(\'At least eight path samples required\')\n        if any(not np.isfinite(v) or v<0 for v in asdict(self).values()):raise ValueError(\'Invalid fit configuration\')\n        if not self.min_width or self.max_width<self.min_width or not 0<self.min_length_ratio<self.max_length_ratio:\n            raise ValueError(\'Invalid width/length limits\')\n\n\ndef _unit(array):\n    array=np.asarray(array,np.float32)\n    if array.ndim!=2 or not np.isfinite(array).all() or array.min()<0 or array.max()>1:\n        raise ValueError(\'Expected a finite 2D ink mask in [0,1]\')\n    return array\n\n\ndef _component(mask,hint=None):\n    threshold=max(.1,min(.4,float(mask.max())*.4))\n    binary=mask>=threshold\n    labels,count=label(binary,np.ones((3,3)))\n    if not count:raise ValueError(\'Empty/low-confidence extracted stroke\')\n    distance=None\n    if hint is not None and hint.max()>.1:\n        distance=distance_transform_edt(hint<max(.1,float(hint.max())*.3))\n    scores=[]\n    for k in range(1,count+1):\n        region=labels==k;mass=float(mask[region].sum())\n        proximity=1. if distance is None else np.exp(-float(np.median(distance[region]))/20.)\n        scores.append(mass*proximity)\n    chosen=labels==(int(np.argmax(scores))+1)\n    return chosen,dict(components=count,selected_mass_fraction=float(mask[chosen].sum()/max(float(mask.sum()),1e-6)))\n\n\ndef _resample(path,n):\n    arc=np.r_[0,np.cumsum(np.linalg.norm(np.diff(path,axis=0),axis=1))]\n    if arc[-1]<1e-6:raise ValueError(\'Degenerate stroke path\')\n    keep=np.r_[True,np.diff(arc)>1e-8];arc=arc[keep];path=path[keep]\n    t=np.linspace(0,arc[-1],n)\n    return np.stack([np.interp(t,arc,path[:,i]) for i in range(2)],1)\n\n\ndef principal_path(binary,n=64):\n    """Longest geodesic path of the principal component; never reject spurs."""\n    skeleton=skeletonize(binary)\n    coordinates=np.argwhere(skeleton)\n    if len(coordinates)<2:raise ValueError(\'Insufficient centerline support\')\n    indices={tuple(yx):i for i,yx in enumerate(coordinates)}\n    graph=[[] for _ in coordinates]\n    for i,(y,x) in enumerate(coordinates):\n        for dy in (-1,0,1):\n            for dx in (-1,0,1):\n                if not (dx or dy):continue\n                j=indices.get((y+dy,x+dx))\n                if j is not None:graph[i].append((j,float(np.hypot(dx,dy))))\n    endpoints=[i for i,edges in enumerate(graph) if len(edges)==1]\n    def farthest(start):\n        distances=np.full(len(graph),np.inf);distances[start]=0.;parent=np.full(len(graph),-1,int)\n        queue=[(0.,start)]\n        while queue:\n            cost,i=heapq.heappop(queue)\n            if cost>distances[i]+1e-8:continue\n            for j,weight in graph[i]:\n                proposed=cost+weight\n                if proposed<distances[j]-1e-8:\n                    distances[j]=proposed;parent[j]=i;heapq.heappush(queue,(proposed,j))\n        candidates=endpoints if endpoints else list(range(len(graph)))\n        end=max(candidates,key=lambda j:distances[j] if np.isfinite(distances[j]) else -1)\n        return end,parent\n    first,_=farthest(endpoints[0] if endpoints else 0)\n    last,parent=farthest(first);order=[last]\n    while order[-1]!=first:\n        previous=parent[order[-1]]\n        if previous<0:raise ValueError(\'Disconnected centerline\')\n        order.append(int(previous))\n    path=coordinates[order,::-1].astype(float)\n    return _resample(path,n),dict(endpoints=len(endpoints),path_coverage=len(order)/len(coordinates),closed=len(endpoints)==0)\n\n\ndef _width(binary,path):\n    radius=distance_transform_edt(binary)\n    samples=map_coordinates(radius,[path[:,1],path[:,0]],order=1,mode=\'nearest\')\n    middle=samples[len(samples)//10:len(samples)-len(samples)//10]\n    return max(1.,float(np.median(2*middle-1)))\n\n\ndef render_reference(reference,original_path,target_path,width_scale):\n    """Render the deformed original centerline with the estimated stroke width.\n\n    Raster thin hints must not be stretched directly: their one-pixel staircase\n    and interpolation profile create serrated, gray thick strokes.\n    """\n    reference=_unit(reference);h,w=reference.shape\n    # Dense samples reduce nearest-segment seams while preserving reference caps.\n    # Keep normalized arc correspondence even when relative segment lengths change.\n    target=np.stack([np.interp(np.linspace(0,1,512),np.linspace(0,1,len(target_path)),target_path[:,i]) for i in range(2)],1)\n    yy,xx=np.mgrid[:h,:w];pixels=np.stack((xx.ravel(),yy.ravel()),1)\n    binary=reference>=max(.1,float(reference.max())*.3)\n    width=_width(binary,original_path)*width_scale\n    distance,_=cKDTree(target).query(pixels)\n    return np.clip(.5+(width/2-distance),0,1).reshape(h,w).astype(np.float32)\n\n\ndef _affine_fallback(reference,observed,config):\n    """Preserve a loop/branch prototype rather than cutting it into a path."""\n    def moments(mask):\n        y,x=np.nonzero(mask);xy=np.stack((x,y),1).astype(float)\n        center=xy.mean(0);extent=np.percentile(xy,95,axis=0)-np.percentile(xy,5,axis=0)\n        return center,np.maximum(extent,1)\n    source=reference>max(.1,float(reference.max())*.3)\n    rc,re=moments(source);tc,te=moments(observed)\n    shift=tc-rc;shift*=min(1.,config.max_displacement/max(np.linalg.norm(shift),1e-6))\n    scale=np.clip(te/re,.75,1.35)\n    # scipy uses y,x and maps output to input.\n    inverse=np.diag(1/scale[::-1]);offset=rc[::-1]-inverse@(rc+shift)[::-1]\n    result=affine_transform(reference,inverse,offset=offset,output_shape=reference.shape,order=1,mode=\'constant\')\n    return result.clip(0,1).astype(np.float32),dict(translation=shift.tolist(),scale=scale.tolist())\n\n\ndef fit_reference_stroke(reference,extracted,*,hint=None,config=None):\n    config=config or ReferenceFitConfig();reference=_unit(reference);extracted=_unit(extracted)\n    if reference.shape!=extracted.shape:raise ValueError(\'Reference/extraction shape mismatch\')\n    hint=reference if hint is None else _unit(hint)\n    if hint.shape!=reference.shape:raise ValueError(\'Hint shape mismatch\')\n    observed,component_info=_component(extracted,hint)\n    canonical,_=_component(reference)\n    original,path_info=principal_path(canonical,config.points)\n    metadata=dict(component_info,reference_path=path_info,warnings=[])\n    if path_info[\'closed\'] or path_info[\'path_coverage\']<.65 or (path_info[\'endpoints\']>2 and path_info[\'path_coverage\']<.9):\n        warped,transform=_affine_fallback(reference,observed,config)\n        metadata.update(method=\'affine_topology_preserving_fallback\',transform=transform)\n        metadata[\'warnings\'].append(\'Closed/branched prototype: affine fallback, no open-curve fit\')\n        return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n            reference_path=None,target_path=None,hint_path=None,width_scale=1.,metadata=metadata)\n    try:\n        hint_binary,_=_component(hint)\n        initial,_=principal_path(hint_binary,config.points)\n        if np.linalg.norm(initial[[0,-1]]-original[[0,-1]])>np.linalg.norm(initial[::-1][[0,-1]]-original[[0,-1]]):initial=initial[::-1]\n        if np.max(np.linalg.norm(initial-original,axis=1))>config.max_displacement:\n            initial=original.copy();metadata[\'warnings\'].append(\'Learned hint displacement too large; initialized from reference\')\n    except ValueError:\n        initial=original.copy();metadata[\'warnings\'].append(\'No usable learned hint; initialized from reference\')\n    target,target_info=principal_path(observed,config.points)\n    if np.linalg.norm(target[[0,-1]]-initial[[0,-1]])>np.linalg.norm(target[::-1][[0,-1]]-initial[[0,-1]]):target=target[::-1]\n    metadata[\'extracted_path\']=target_info\n    if target_info[\'closed\']:\n        target=initial.copy();metadata[\'warnings\'].append(\'Closed extracted path for open reference: retained initialization\')\n    if target_info[\'path_coverage\']<.75:metadata[\'warnings\'].append(\'Side branches excluded by principal-path fit\')\n    if component_info[\'selected_mass_fraction\']<.65:metadata[\'warnings\'].append(\'Large disconnected extraction mass excluded\')\n    chord=original[-1]-original[0];length=np.linalg.norm(np.diff(original,axis=0),axis=1).sum()\n    normal=np.array([-chord[1],chord[0]])/max(np.linalg.norm(chord),1e-6)\n    deviation=np.max(np.abs((original-original[0])@normal))\n    straight=deviation<=1.1 or deviation/max(length,1)<.025\n    if straight:\n        # Robust TLS: trim off-axis centerline points before estimating direction.\n        points=target.copy()\n        for _ in range(3):\n            center=points.mean(0);_,_,v=np.linalg.svd(points-center,full_matrices=False);axis=v[0]\n            residual=np.abs((target-center)@np.array([-axis[1],axis[0]]))\n            points=target[residual<=max(.75,float(np.quantile(residual,.85)))]\n        projection=(target-center)@axis\n        if axis@(initial[-1]-initial[0])<0:axis=-axis;projection=-projection\n        endpoints=np.quantile(projection,[.01,.99])\n        target=center+np.linspace(*endpoints,config.points)[:,None]*axis\n        metadata[\'method\']=\'robust_straight_reference_fit\'\n    else:metadata[\'method\']=\'regularized_curved_reference_fit\'\n    target_length=np.linalg.norm(np.diff(target,axis=0),axis=1).sum()\n    ratio=target_length/max(length,1e-6)\n    if not config.min_length_ratio<=ratio<=config.max_length_ratio:\n        metadata[\'warnings\'].append(\'Extracted path length outside allowed range; retained bounded initialization\')\n        target=initial.copy()\n        if straight:target=np.linspace(target[0],target[-1],config.points)\n    n=config.points;second=np.diff(np.eye(n),n=2,axis=0)\n    system=(1+config.shape_prior)*np.eye(n)+config.smoothness*(second.T@second)\n    if straight:\n        fitted=target\n    else:\n        # Smooth the deformation, rather than tracing every observed bend/spur.\n        fitted=initial+np.linalg.solve(system,target-initial)\n        endpoints=fitted[[0,-1]].copy()\n        fitted=gaussian_filter1d(fitted,.8,axis=0,mode=\'nearest\')\n        fitted[0],fitted[-1]=endpoints\n    displacement=fitted-original\n    norm=np.linalg.norm(displacement,axis=1,keepdims=True)\n    limit=min(1.,config.max_displacement/max(float(norm.max()),1e-6)) if straight else np.minimum(1,config.max_displacement/np.maximum(norm,1e-6))\n    fitted=original+displacement*limit\n    ref_width=_width(canonical,original)\n    target_width=float(np.clip(_width(observed,target),config.min_width,config.max_width))\n    width_scale=float(np.clip(target_width/ref_width,.5,8.))\n    warped=render_reference(reference,original,fitted,width_scale)\n    metadata.update(width_reference=ref_width,width_observed=target_width,width_scale=width_scale,\n        extracted_length_ratio=float(ratio),max_displacement=float(np.linalg.norm(fitted-original,axis=1).max()))\n    return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n        reference_path=original,target_path=fitted,hint_path=initial,width_scale=width_scale,metadata=metadata)\n\n\ndef fit_reference_glyph(strokes,extracted,*,hints=None,glyph_reference=None,config=None):\n    config=config or ReferenceFitConfig()\n    strokes=np.asarray(strokes,np.float32);extracted=np.asarray(extracted,np.float32)\n    if strokes.shape!=extracted.shape or strokes.ndim!=3:raise ValueError(\'K matching reference/extracted masks required\')\n    hints=strokes if hints is None else np.asarray(hints,np.float32)\n    if hints.shape!=strokes.shape:raise ValueError(\'Hint shape mismatch\')\n    fitted=[]\n    for k in range(len(strokes)):\n        try:item=fit_reference_stroke(strokes[k],extracted[k],hint=hints[k],config=config)\n        except ValueError as exc:\n            item=dict(warped=strokes[k].copy(),hint=hints[k],reference=strokes[k],extracted=extracted[k],\n                reference_path=None,target_path=None,hint_path=None,width_scale=1.,\n                metadata=dict(method=\'unreliable_reference_fallback\',warnings=[str(exc)]))\n        fitted.append(item)\n    # Joint bounded translations keep reference contacts from independently drifting.\n    matrix=np.eye(len(strokes));rhs=np.zeros((len(strokes),2));contacts=[]\n    for i,a in enumerate(fitted):\n        if a[\'reference_path\'] is None:continue\n        for j in range(i+1,len(fitted)):\n            b=fitted[j]\n            if b[\'reference_path\'] is None:continue\n            distance=np.linalg.norm(a[\'reference_path\'][:,None]-b[\'reference_path\'][None],axis=2)\n            u,v=np.unravel_index(distance.argmin(),distance.shape)\n            connected=distance[u,v]<=config.junction_radius\n            if not connected and glyph_reference is not None and distance[u,v]<=10:\n                line=np.linspace(a[\'reference_path\'][u],b[\'reference_path\'][v],32)\n                ink=map_coordinates(_unit(np.asarray(glyph_reference).squeeze()),[line[:,1],line[:,0]],order=1,mode=\'constant\')\n                connected=float((ink>.25).mean())>.85\n            if not connected:continue\n            # Sparse annotations often omit the segment inside a junction. If\n            # the full reference confirms a connection, extend an endpoint to\n            # the other stroke instead of translating its whole visible tip.\n            if glyph_reference is not None:\n                ia=0 if u<4 else -1 if u>=len(a[\'reference_path\'])-4 else None\n                ib=0 if v<4 else -1 if v>=len(b[\'reference_path\'])-4 else None\n                if (ia is None)!=(ib is None):\n                    end_item,end_index,other,other_index=(a,ia,b,v) if ia is not None else (b,ib,a,u)\n                    difference=other[\'target_path\'][other_index]-end_item[\'target_path\'][end_index]\n                    gap=float(np.linalg.norm(difference))\n                    if gap<=config.max_displacement:\n                        t=np.linspace(0,1,len(end_item[\'target_path\']))\n                        factor=(1-t)**3 if end_index==0 else t**3\n                        if end_item[\'metadata\'].get(\'method\')==\'robust_straight_reference_fit\':factor=(1-t) if end_index==0 else t\n                        end_item[\'target_path\']=end_item[\'target_path\']+factor[:,None]*difference\n                        end_item[\'metadata\'].setdefault(\'endpoint_extensions\',[]).append(gap)\n            difference=a[\'target_path\'][u]-b[\'target_path\'][v]\n            weight=config.junction_weight\n            matrix[i,i]+=weight;matrix[j,j]+=weight;matrix[i,j]-=weight;matrix[j,i]-=weight\n            rhs[i]-=weight*difference;rhs[j]+=weight*difference\n            contacts.append(dict(strokes=[i,j],indices=[int(u),int(v)],gap_before=float(np.linalg.norm(difference))))\n    shift=np.linalg.solve(matrix,rhs)\n    shift*=np.minimum(1,config.max_junction_shift/np.maximum(np.linalg.norm(shift,axis=1,keepdims=True),1e-6))\n    for i,item in enumerate(fitted):\n        if item[\'target_path\'] is None:continue\n        item[\'target_path\']=item[\'target_path\']+shift[i]\n        displacement=item[\'target_path\']-item[\'reference_path\']\n        maximum=float(np.linalg.norm(displacement,axis=1).max())\n        if maximum>config.max_displacement:\n            item[\'target_path\']=item[\'reference_path\']+displacement*(config.max_displacement/maximum)\n            item[\'metadata\'][\'warnings\'].append(\'Joint displacement clipped; inspect contact consistency\')\n        item[\'warped\']=render_reference(item[\'reference\'],item[\'reference_path\'],item[\'target_path\'],item[\'width_scale\'])\n        item[\'metadata\'][\'junction_translation\']=shift[i].tolist()\n        item[\'metadata\'][\'final_max_displacement\']=float(np.linalg.norm(item[\'target_path\']-item[\'reference_path\'],axis=1).max())\n    for contact in contacts:\n        i,j=contact[\'strokes\'];u,v=contact[\'indices\']\n        contact[\'gap_after\']=float(np.linalg.norm(fitted[i][\'target_path\'][u]-fitted[j][\'target_path\'][v]))\n    masks=np.stack([item[\'warped\'] for item in fitted])\n    return dict(strokes=masks,candidate=masks.max(0),fits=fitted,contacts=contacts,config=asdict(config))\n', 'diffusion_model/selectable_adaptive_sweep.py': '"""Select four references, generate independent cases, fit and repair all cases.\n\nThe notebook supplies the historical adaptive runtime. This module orchestrates\nexperiments and records failures; it does not alter its losses or accept updates.\n"""\nimport gc\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\n\nfrom .score_pilot import training_catalog, load_catalog_reference\nfrom .multiglyph_pilot import compose_sheet, generate_multiglyph_pilot\nfrom .fitted_candidate_diffusion import image_tensor, run_pair\nfrom stroke_model.stroke_reference_fit import fit_reference_glyph, ReferenceFitConfig\nfrom stroke_model.adaptive_ink_constraints import AdaptiveInkConstraints\n\n\nADAPTIVE_COMMIT = \'d74e26e19d1e13360fdecde5d9d80d889344d990\'\n\n\ndef select_entries(manifest, rows):\n    rows = list(rows)\n    catalog = {r[\'manifest_row\']: r for r in training_catalog(manifest)}\n    if len(rows) != 4 or len(set(rows)) != 4 or any(type(i) is not int or i not in catalog for i in rows):\n        raise ValueError(\'Choose exactly four distinct row IDs shown in the reference catalog\')\n    return [catalog[i] for i in rows]\n\n\ndef prepare_groups(manifest, selections, output):\n    """Export only selected canonical references; retain original row identities."""\n    choices = [list(rows) for rows in selections]\n    if not choices or len({tuple(r) for r in choices}) != len(choices):\n        raise ValueError(\'Choose at least one group, without duplicate ordered groups\')\n    entries = [select_entries(manifest, rows) for rows in choices]\n    output = Path(output);output.mkdir(parents=True, exist_ok=False)\n    groups = []\n    for i, (rows, selected) in enumerate(zip(choices, entries)):\n        root = output/f\'group_{i:03d}\';conditions = root/\'source_conditions\'\n        conditions.mkdir(parents=True)\n        local = []\n        for slot, entry in enumerate(selected):\n            reference, strokes = load_catalog_reference(entry)\n            name = f\'condition_{slot+1}.npz\'\n            np.savez_compressed(conditions/name, reference=reference, strokes=strokes)\n            local.append(dict(path=name, split=\'train\', char_id=entry.get(\'char_id\', \'\'),\n                              original_manifest_row=rows[slot]))\n        path = conditions/\'conditions.json\'\n        path.write_text(json.dumps(local, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        if len(training_catalog(path)) != 4:\n            raise ValueError(\'Selected rows contain identical reference/stroke contents; choose four different glyphs\')\n        group = dict(group_id=f\'group_{i:03d}\', original_rows=rows,\n                     char_ids=[r.get(\'char_id\', \'\') for r in selected],\n                     manifest=str(path), root=str(root))\n        groups.append(group)\n    (output/\'groups.json\').write_text(json.dumps(groups, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    return groups\n\n\ndef generate_cases(groups, *, strengths=(.65,), seeds=(100,),\n                   steps=100, canvas_size=576, model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\',\n                   revision=None, pipeline=None, include_control=False):\n    cases = []\n    for group in groups:\n        root = Path(group[\'root\'])\n        try:\n            report = generate_multiglyph_pilot(group[\'manifest\'], [1, 2, 3, 4], root/\'generation\',\n                glyph_counts=(4,), canvas_size=canvas_size, strengths=strengths, seeds=seeds,\n                steps=steps, model_id=model_id, revision=revision, pipeline=pipeline)\n        except Exception as exc:\n            (root/\'generation_failure.json\').write_text(json.dumps(dict(type=type(exc).__name__, error=str(exc))), encoding=\'utf-8\')\n            partial = root/\'generation/generation_report.json\'\n            if not partial.is_file():continue\n            report = json.loads(partial.read_text(encoding=\'utf-8\'))\n        for page in report[\'pages\']:\n            if page[\'control\'] and not include_control:continue\n            case_id = f"{group[\'group_id\']}_page_{page[\'page_id\']:03d}"\n            cases.append(dict(case_id=case_id, group_id=group[\'group_id\'], group=group,\n                input=page[\'input\'], control=page[\'control\'], strength=page[\'strength\'], seed=page[\'seed\'],\n                safety_filtered=page.get(\'safety_filtered\'), label=page[\'label\'],\n                prompt=report.get(\'prompts\', {}).get(\'4\', \'\'), negative_prompt=report.get(\'negative_prompt\', \'\'),\n                output=str(root/case_id), status=\'generated\'))\n    return cases\n\n\ndef validate_generation_cases(cases, *, seed, strength):\n    """Changing a generation seed/strength requires fresh stage-1 images."""\n    if not cases or any(c[\'control\'] or c[\'seed\'] != seed or c[\'strength\'] != strength for c in cases):\n        raise ValueError(\'생성 조건이 다릅니다. 4번 셀에서 해당 seed/strength로 새로 생성한 뒤 5번을 실행하세요.\')\n\n\ndef fit_case(case, extractor):\n    out = Path(case[\'output\']);out.mkdir(parents=True, exist_ok=False)\n    with Image.open(case[\'input\']) as im:source = im.convert(\'RGB\').copy()\n    entries = training_catalog(case[\'group\'][\'manifest\'])\n    _, slots = compose_sheet(entries, source.width)\n    if source.width != source.height:raise ValueError(\'Expected square joint sheet\')\n    device = next(extractor.parameters(), torch.tensor(0.)).device\n    rgb = image_tensor(source, device)\n    ink = 1-(rgb*rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n    records = []\n    candidate = Image.new(\'RGB\', source.size, \'white\')\n    source_array = np.asarray(source, np.float32)/255\n    for i, slot in enumerate(slots):\n        x0, y0, x1, y1 = slot[\'bbox\']\n        reference = torch.as_tensor(slot[\'reference\'], device=device, dtype=torch.float32)[None]\n        strokes = torch.as_tensor(slot[\'strokes\'], device=device, dtype=torch.float32)[None]\n        crop = F.interpolate(ink[..., y0:y1, x0:x1], strokes.shape[-2:], mode=\'bilinear\', align_corners=False)\n        with torch.no_grad():prediction = extractor(crop, reference, strokes)\n        masks = prediction[\'masks\'][0].cpu().numpy()\n        hints = prediction.get(\'hints\', strokes)[0].cpu().numpy()\n        result = fit_reference_glyph(slot[\'strokes\'], masks, hints=hints, glyph_reference=slot[\'reference\'],\n            config=ReferenceFitConfig(max_displacement=16., smoothness=12., shape_prior=.15, max_junction_shift=2.))\n        records.append(dict(reference=slot[\'reference\'], strokes=slot[\'strokes\'], fitted=result[\'strokes\'],\n            extracted_masks=masks,fit_metadata=[r[\'metadata\'] for r in result[\'fits\']]))\n        np.savez_compressed(out/f\'warped_slot_{i+1}.npz\', warped_strokes=result[\'strokes\'],\n                            reference_strokes=slot[\'strokes\'], extracted_masks=masks,\n                            probabilities=prediction[\'probabilities\'][0].cpu().numpy(), model_hints=hints)\n        (out/f\'fit_slot_{i+1}.json\').write_text(json.dumps(dict(strokes=[r[\'metadata\'] for r in result[\'fits\']], contacts=result[\'contacts\']), ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        pixel_crop = source_array[y0:y1, x0:x1];brightness = pixel_crop.mean(-1)\n        background = np.median(pixel_crop[brightness >= np.quantile(brightness, .8)], axis=0)\n        foreground = np.median(pixel_crop[brightness <= np.quantile(brightness, .05)], axis=0)\n        alpha = np.asarray(Image.fromarray(result[\'candidate\'].astype(np.float32)).resize((x1-x0, y1-y0), Image.Resampling.BICUBIC)).clip(0, 1)\n        patch = background[None, None]*(1-alpha[..., None])+foreground[None, None]*alpha[..., None]\n        candidate.paste(Image.fromarray(np.rint(patch*255).clip(0, 255).astype(\'uint8\')), (x0, y0))\n    source.save(out/\'input.png\');candidate.save(out/\'fitted_candidate.png\')\n    (out/\'case.json\').write_text(json.dumps(case, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    return source, candidate, records\n\n\ndef repair_cases(cases, extractor, *, checkpoint_sha256, revision, seed=2026,\n                 steps=60, strength=.25, on_case=None, progress=None):\n    """Run raw historical adaptive on all cases; failures remain explicit rows."""\n    summaries = []\n    for case in cases:\n        row = {k: case[k] for k in (\'case_id\', \'group_id\', \'control\', \'strength\', \'seed\', \'safety_filtered\', \'input\', \'output\')}\n        try:\n            source, candidate, records = fit_case(case, extractor)\n            source_rgb = image_tensor(source, next(extractor.parameters(), torch.tensor(0.)).device)\n            objective = AdaptiveInkConstraints(source_rgb, extractor, records,\n                refresh_every=3, enable_restore=False, margin_ratio=.20)\n            metadata = dict(case={k: v for k, v in case.items() if k != \'group\'},\n                checkpoint_sha256=checkpoint_sha256, original_manifest_rows=case[\'group\'][\'original_rows\'],\n                prompt=case[\'prompt\'], negative_prompt=case[\'negative_prompt\'],\n                candidate_is_ground_truth=False, adaptive_runtime_commit=ADAPTIVE_COMMIT)\n            report = run_pair(source, candidate, metadata, Path(case[\'output\'])/\'reverse\',\n                steps=steps, strength=strength, seed=seed, inner_steps=4, step_ratio=.012,\n                revision=revision, progress=progress, guidance_mode=\'adaptive_ink\', objective_override=objective)\n            if report.get(\'selection\') is not None:raise RuntimeError(\'Unexpected release selection: historical runtime required\')\n            row.update(status=\'completed\', guided=str(Path(case[\'output\'])/\'reverse/adaptive_guided.png\'),\n                baseline=str(Path(case[\'output\'])/\'reverse/redenoise_only.png\'),\n                candidate=str(Path(case[\'output\'])/\'fitted_candidate.png\'), metrics=report[\'metrics\'])\n        except Exception as exc:\n            row.update(status=\'failed\', error_type=type(exc).__name__, error=str(exc))\n            Path(case[\'output\']).mkdir(parents=True, exist_ok=True)\n            (Path(case[\'output\'])/\'failure.json\').write_text(json.dumps(row, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        summaries.append(row)\n        summary_path = Path(case[\'group\'][\'root\']).parent/\'all_results.json\'\n        summary_path.write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        if on_case:on_case(row)\n        gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n    return summaries\n', 'diffusion_model/score_pilot.py': '"""Pretrained img2img outputs on training references, then unchanged source scores.\n\nThis is a domain-transfer pilot, not text-to-image accuracy or guidance training.\n"""\nfrom datetime import datetime\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\n\ndef load_catalog_reference(entry):\n    """Read one selected glyph, from either a training NPZ or reference PNGs."""\n    if entry.get(\'path\'):\n        with np.load(entry[\'path\'],allow_pickle=False) as data:\n            reference,strokes=[data[key].astype(np.float32) for key in (\'reference\',\'strokes\')]\n    else:\n        from stroke_model.convert_reference_csv import load_ink\n        reference=load_ink(entry[\'reference\'],entry[\'image_size\'])\n        strokes=np.concatenate([load_ink(p,entry[\'image_size\']) for p in entry[\'reference_strokes\']])\n    if (reference.ndim!=3 or reference.shape[0]!=1 or reference.shape[1]!=reference.shape[2]\n            or strokes.ndim!=3 or strokes.shape[1:]!=reference.shape[1:] or not len(strokes)):\n        raise ValueError(\'Invalid reference/stroke shapes\')\n    for array in (reference,strokes):\n        if not np.isfinite(array).all() or array.min()<0 or array.max()>1:\n            raise ValueError(\'Expected finite absolute ink in [0,1]\')\n    if reference.max()<=.5 or (strokes.max(axis=(1,2))<=.5).any():\n        raise ValueError(\'Empty reference or stroke\')\n    return reference,strokes\n\n\ndef training_catalog(manifest):\n    """Only explicit train rows; never infer characters from numeric image IDs."""\n    manifest = Path(manifest).resolve()\n    rows = json.loads(manifest.read_text(encoding=\'utf-8\'))\n    if not isinstance(rows, list):\n        raise ValueError(\'Expected the stroke-training manifest JSON list\')\n    groups = {}\n    catalog, seen = [], set()\n    for index, row in enumerate(rows):\n        split = row.get(\'split\')\n        if split not in (\'train\', \'val\', \'test\'):\n            raise ValueError(\'Every manifest row needs train/val/test split\')\n        if row.get(\'group_id\'):\n            previous = groups.setdefault(row[\'group_id\'], split)\n            if previous != split:\n                raise ValueError(\'Training manifest contains group leakage\')\n        if split != \'train\':\n            continue\n        if row.get(\'format\') == \'reference_paths_v1\':\n            digest=row[\'reference_sha256\']\n            if digest in seen:continue\n            seen.add(digest)\n            catalog.append({**row,\'manifest_row\':index+1})\n            continue\n        path = (manifest.parent / row[\'path\']).resolve()\n        with np.load(path, allow_pickle=False) as data:\n            reference, strokes = [data[key].astype(np.float32) for key in (\'reference\', \'strokes\')]\n        if (reference.ndim != 3 or reference.shape[0] != 1 or reference.shape[1] != reference.shape[2]\n                or strokes.ndim != 3 or strokes.shape[1:] != reference.shape[1:] or not len(strokes)):\n            raise ValueError(f\'Invalid reference/strokes shapes: {path}\')\n        for array in (reference, strokes):\n            if not np.isfinite(array).all() or array.min() < 0 or array.max() > 1:\n                raise ValueError(f\'Expected finite absolute ink in [0,1]: {path}\')\n        if reference.max() <= .5 or (strokes.max(axis=(1, 2)) <= .5).any():\n            raise ValueError(f\'Empty reference or stroke: {path}\')\n        digest = hashlib.sha256(reference.tobytes() + strokes.tobytes()).hexdigest()\n        if digest in seen:\n            continue\n        seen.add(digest)\n        catalog.append(dict(manifest_row=index + 1, path=str(path),\n                            char_id=str(row.get(\'char_id\', \'\')), reference_sha256=digest))\n    if not catalog:\n        raise ValueError(\'No train references in this manifest\')\n    return catalog\n\n\ndef scored_ink(image, size):\n    """Fixed black-on-white luminance conversion; no polarity guesses or cropping."""\n    rgba = image.convert(\'RGBA\')\n    white = Image.new(\'RGBA\', rgba.size, (255, 255, 255, 255))\n    gray = Image.alpha_composite(white, rgba).convert(\'L\')\n    gray = gray.resize((size, size), Image.Resampling.BILINEAR)\n    return (1 - np.asarray(gray, dtype=np.float32) / 255)[None].copy()\n\n\ndef generate_pilot(manifest, manifest_rows, output, *, strengths=(.25, .45, .65),\n                   seeds=(100, 101), steps=30,\n                   model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\', revision=None,\n                   pipeline=None):\n    """Save every output, including wrong letters, noise and safety-filter flags.\n\n    pipeline injection is for orchestration tests, never evidence of SD quality.\n    The caller must supply the manifest used for the selected stroke checkpoint.\n    """\n    catalog = {entry[\'manifest_row\']: entry for entry in training_catalog(manifest)}\n    selected = list(dict.fromkeys(manifest_rows))\n    if not selected or any(type(i) is not int or i not in catalog for i in selected):\n        raise ValueError(\'Select only manifest row numbers shown in the train catalog\')\n    strengths, seeds = list(strengths), list(seeds)\n    if (type(steps) is not int or steps < 2 or not strengths or not seeds\n            or any(not np.isfinite(s) or not 0 < s < 1 or int(steps*s) < 1 for s in strengths)\n            or any(type(s) is not int or s < 0 for s in seeds)):\n        raise ValueError(\'Need valid strengths, integer seeds and nonempty schedules\')\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    import torch\n    try:\n        diffusers_version = importlib.metadata.version(\'diffusers\')\n    except importlib.metadata.PackageNotFoundError:\n        diffusers_version = None\n    if pipeline is None:\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Run generation on a Colab GPU; local CPU generation is not started\')\n        from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n        kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n        if revision:\n            kwargs[\'revision\'] = revision\n        pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n        pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)\n        pipeline.enable_attention_slicing()\n        pipeline.to(\'cuda\')\n    destination.mkdir(parents=True, exist_ok=False)\n    prompt = (\'a single isolated Korean Hangul glyph, black ink, plain white background, \'\n              \'flat front view, clean printed typography\')\n    negative = \'photograph, scene, paper texture, shadows, colored background, decorations\'\n    report = dict(experiment=\'pretrained_img2img_domain_pilot\', free_text_generation=False,\n                  guidance_applied=False, model_id=model_id, requested_revision=revision,\n                  pipeline_class=type(pipeline).__name__,\n                  resolved_commit=getattr(pipeline, \'config\', {}).get(\'_commit_hash\'),\n                  scheduler=dict(pipeline.scheduler.config),\n                  torch_version=torch.__version__, diffusers_version=diffusers_version,\n                  training_manifest=str(Path(manifest).resolve()),\n                  training_manifest_sha256=hashlib.sha256(Path(manifest).read_bytes()).hexdigest(),\n                  training_manifest_checkpoint_link=\'caller must use the actual checkpoint training manifest\',\n                  prompt=prompt, negative_prompt=negative, steps=steps, strengths=strengths,\n                  seeds=seeds, preprocessing=\'fixed 1-luminance; bilinear resize; no crop/threshold/repair\',\n                  results=[])\n\n    def save(image, entry, reference, strokes, label, **metadata):\n        prefix = destination / f\'sample_{len(report["results"]):04d}\'\n        image.save(str(prefix) + \'_raw.png\')\n        ink = scored_ink(image, reference.shape[-1])\n        Image.fromarray(np.uint8(np.clip(1-ink[0], 0, 1)*255)).save(str(prefix)+\'_scored.png\')\n        np.savez_compressed(str(prefix)+\'.npz\', image=ink, reference=reference, reference_strokes=strokes)\n        report[\'results\'].append(dict(**entry, label=label, input=str(prefix)+\'_raw.png\',\n                                     npz=str(prefix)+\'.npz\', **metadata))\n        (destination/\'generation_report.json\').write_text(\n            json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n    for row in selected:\n        entry = catalog[row]\n        reference,strokes=load_catalog_reference(entry)\n        initial = Image.fromarray(np.uint8((1-reference[0])*255)).convert(\'RGB\')\n        save(initial, entry, reference, strokes, \'reference_control\', control=True)\n        initial = initial.resize((512, 512), Image.Resampling.BICUBIC)\n        initial.save(destination/f\'row_{row}_initial_512.png\')\n        for strength in strengths:\n            for seed in seeds:\n                print(f\'Generate train row={row}, strength={strength}, seed={seed}\', flush=True)\n                result = pipeline(prompt=prompt, negative_prompt=negative, image=initial,\n                                  strength=float(strength), num_inference_steps=steps,\n                                  guidance_scale=7.5, generator=torch.Generator(\'cpu\').manual_seed(seed))\n                flags = getattr(result, \'nsfw_content_detected\', None)\n                save(result.images[0], entry, reference, strokes, f\'strength={strength} seed={seed}\',\n                     control=False, strength=float(strength), seed=seed,\n                     safety_filtered=bool(flags[0]) if flags is not None else None)\n    return report\n\n\ndef score_pilot(generation_report, checkpoint, *, score_config=None, source_config=None, show=True):\n    """Load the frozen existing extractor only AFTER the generator is released."""\n    import torch\n    from stroke_model.infer_topology import load_topology\n    from stroke_model.topology_scoring import score_topology_results\n    report = json.loads(Path(generation_report).read_text(encoding=\'utf-8\'))\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model, size, epoch = load_topology(checkpoint, device)\n    extraction = dict(checkpoint=str(checkpoint), checkpoint_epoch=epoch,\n                      output_mode=getattr(model, \'output_mode\', \'independent\'), results=[])\n    root = Path(generation_report).parent / (\'extraction_\'+datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    root.mkdir(exist_ok=False)\n    for index, sample in enumerate(report[\'results\']):\n        with np.load(sample[\'npz\'], allow_pickle=False) as data:\n            arrays = {key: data[key].copy() for key in (\'image\', \'reference\', \'reference_strokes\')}\n        if arrays[\'reference\'].shape != (1, size, size):\n            raise ValueError(\'Training reference resolution differs from checkpoint; do not rescale strokes\')\n        with torch.inference_mode():\n            predicted = model(*[torch.from_numpy(arrays[k])[None].to(device)\n                                for k in (\'image\', \'reference\', \'reference_strokes\')])\n        arrays.update({k: predicted[v][0].float().cpu().numpy()\n                       for k, v in ((\'soft_masks\', \'masks\'), (\'probabilities\', \'probabilities\'))})\n        path = root/f\'sample_{index:04d}.npz\'\n        np.savez_compressed(path, **arrays)\n        extraction[\'results\'].append({**sample, \'npz\': str(path)})\n    del model\n    result = score_topology_results(extraction, score_config=score_config, source_config=source_config,\n                                   save_dir=root/\'scores\', show=show, export_debug=True)\n    import pandas as pd\n    rows = []\n    for sample, measured in zip(extraction[\'results\'], result[\'indicator\'][\'results\']):\n        rows.append({key: sample.get(key) for key in\n                     (\'manifest_row\', \'char_id\', \'label\', \'control\', \'strength\', \'seed\', \'safety_filtered\',\n                      \'page_id\', \'glyph_count\', \'slot\', \'crop_bbox\')}\n                    | {key: measured[key] for key in (\'structural_score\', \'missing_score_cap\',\n                       \'split_score_cap\', \'junction_score_cap\', \'extra_score_cap\')})\n    pd.DataFrame(rows).to_csv(root/\'pilot_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    import zipfile\n    with zipfile.ZipFile(result[\'debug_zip\'], \'a\', zipfile.ZIP_DEFLATED) as archive:\n        archive.write(generation_report, \'generation_report.json\')\n        archive.write(root/\'pilot_scores.csv\', \'pilot_scores.csv\')\n        for index, sample in enumerate(report[\'results\']):\n            raw = Path(sample[\'input\'])\n            archive.write(raw, f\'generated_originals/{index:04d}.png\')\n        for page in report.get(\'pages\', []):\n            archive.write(page[\'input\'], f\'generated_sheets/{page["page_id"]:03d}.png\')\n    return result, pd.DataFrame(rows)\n', 'diffusion_model/multiglyph_pilot.py': '"""One jointly denoised glyph sheet, evaluated with fixed per-slot crops."""\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .score_pilot import training_catalog, scored_ink, load_catalog_reference\n\n\ndef compose_sheet(entries, canvas_size=576, cell_pixels=None):\n    """Preserve the original full reference canvas inside each square slot."""\n    side = math.isqrt(len(entries))\n    if side*side != len(entries) or side < 1:\n        raise ValueError(\'Use square glyph counts: 1, 4, 9, 16, ...\')\n    pixels = canvas_size//side if cell_pixels is None else cell_pixels\n    if type(pixels) is not int or pixels < 16 or (pixels*side) % 8:\n        raise ValueError(\'Sheet dimensions must be multiples of 8; cells need at least 16 pixels\')\n    if cell_pixels is None and canvas_size % side:\n        raise ValueError(\'CANVAS_SIZE must be divisible by every selected grid side\')\n    sheet = Image.new(\'RGB\', (pixels*side, pixels*side), \'white\')\n    slots = []\n    for index, entry in enumerate(entries):\n        reference,strokes=load_catalog_reference(entry)\n        glyph = Image.fromarray(np.uint8((1-reference[0])*255)).convert(\'RGB\')\n        x, y = (index % side)*pixels, (index//side)*pixels\n        bbox = [x, y, x+pixels, y+pixels]\n        sheet.paste(glyph.resize((pixels, pixels), Image.Resampling.BICUBIC), (x, y))\n        slots.append(dict(entry=entry, reference=reference, strokes=strokes, bbox=bbox, slot=index))\n    return sheet, slots\n\n\ndef generate_multiglyph_pilot(manifest, manifest_rows, output, *, glyph_counts=(1, 4, 9),\n                             canvas_size=576, cell_pixels=None, strengths=(.65, .85),\n                             seeds=(100,), steps=30,\n                             model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\',\n                             revision=None, pipeline=None):\n    """A pipeline call sees the ENTIRE sheet. Never generate tiles separately.\n\n    Crops use input slot coordinates, never OCR success or connected components.\n    Missing, displaced, merged and incorrect glyphs remain in the output set.\n    """\n    catalog = {item[\'manifest_row\']: item for item in training_catalog(manifest)}\n    selected = list(dict.fromkeys(manifest_rows))\n    counts, strengths, seeds = list(glyph_counts), list(strengths), list(seeds)\n    if not selected or any(type(i) is not int or i not in catalog for i in selected):\n        raise ValueError(\'Select only reference rows in the training catalog\')\n    if (not counts or any(type(n) is not int or n < 1 or math.isqrt(n)**2 != n for n in counts)\n            or len(set(counts)) != len(counts) or max(counts) > len(selected)):\n        raise ValueError(\'Select enough training references for glyph counts 1/4/9/16/...\')\n    if (type(steps) is not int or steps < 2 or not strengths or not seeds\n            or any(not np.isfinite(s) or not 0 < s < 1 or int(steps*s) < 1 for s in strengths)\n            or any(type(s) is not int or s < 0 for s in seeds)):\n        raise ValueError(\'Invalid strength, seed or step schedule\')\n    entries = [catalog[row] for row in selected]\n    # Validate layout and references before downloading weights/creating outputs.\n    layouts = [(count, *compose_sheet(entries[:count], canvas_size, cell_pixels)) for count in counts]\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    import torch\n    if pipeline is None:\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Generate on a Colab GPU\')\n        from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n        kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n        if revision:\n            kwargs[\'revision\'] = revision\n        pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n        pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)\n        pipeline.enable_attention_slicing()\n        pipeline.to(\'cuda\')\n    destination.mkdir(parents=True, exist_ok=False)\n    try:\n        version = importlib.metadata.version(\'diffusers\')\n    except importlib.metadata.PackageNotFoundError:\n        version = None\n    report = dict(experiment=\'joint_sheet_img2img_fixed_crop\', free_text_generation=False,\n        guidance_applied=False, model_id=model_id, requested_revision=revision,\n        pipeline_class=type(pipeline).__name__, scheduler=dict(pipeline.scheduler.config),\n        torch_version=torch.__version__, diffusers_version=version,\n        training_manifest=str(Path(manifest).resolve()),\n        training_manifest_sha256=hashlib.sha256(Path(manifest).read_bytes()).hexdigest(),\n        selected_manifest_rows=selected, glyph_counts=counts,\n        pixel_budget=\'fixed_sheet\' if cell_pixels is None else \'fixed_cell\',\n        requested_canvas_size=canvas_size, cell_pixels=cell_pixels,\n        strengths=strengths, seeds=seeds, steps=steps,\n        crop_policy=\'fixed input slot coordinates; no OCR filtering, recentering or repair\',\n        comparison_note=\'Counts use nested prefixes; only shared reference rows are paired. \'\n                        \'Fixed-sheet mode also changes pixels per glyph. More errors are not guaranteed.\',\n        results=[], pages=[])\n\n    def save_page(image, slots, count, *, control=False, strength=None, seed=None, flags=None):\n        page_id = len(report[\'pages\'])\n        path = destination/f\'page_{page_id:03d}.png\'\n        image.save(path)\n        label = f\'{count} glyphs | \' + (\'input control\' if control else f\'strength={strength} seed={seed}\')\n        page = dict(page_id=page_id, input=str(path), glyph_count=count, control=control,\n                    strength=strength, seed=seed, width=image.width, height=image.height, label=label,\n                    safety_filtered=flags, slots=[])\n        for slot in slots:\n            prefix = destination/f\'sample_{len(report["results"]):04d}\'\n            crop = image.crop(tuple(slot[\'bbox\']))\n            crop.save(str(prefix)+\'_raw.png\')\n            ink = scored_ink(crop, slot[\'reference\'].shape[-1])\n            Image.fromarray(np.uint8(np.clip(1-ink[0],0,1)*255)).save(str(prefix)+\'_scored.png\')\n            np.savez_compressed(str(prefix)+\'.npz\', image=ink,\n                                reference=slot[\'reference\'], reference_strokes=slot[\'strokes\'])\n            sample = dict(**slot[\'entry\'], page_id=page_id, page_input=str(path), glyph_count=count,\n                slot=slot[\'slot\'], crop_bbox=slot[\'bbox\'], control=control, strength=strength, seed=seed,\n                safety_filtered=flags, label=label+f\' | slot={slot["slot"]+1}\',\n                input=str(prefix)+\'_raw.png\', npz=str(prefix)+\'.npz\')\n            report[\'results\'].append(sample)\n            page[\'slots\'].append(dict(slot=slot[\'slot\'], bbox=slot[\'bbox\'],\n                                      manifest_row=slot[\'entry\'][\'manifest_row\']))\n        report[\'pages\'].append(page)\n        (destination/\'generation_report.json\').write_text(\n            json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n    for count, initial, slots in layouts:\n        side = math.isqrt(count)\n        save_page(initial, slots, count, control=True)\n        prompt = (f\'a typography sheet with {count} Korean Hangul glyphs arranged in \'\n                  f\'{side} rows and {side} columns, black ink on plain white background, \'\n                  \'flat front view, no grid lines, preserve the glyphs and layout of the input image\')\n        report.setdefault(\'prompts\', {})[str(count)] = prompt\n        negative = \'photograph, scenery, paper texture, shadows, colored background, grid lines, decorations\'\n        report[\'negative_prompt\'] = negative\n        for strength in strengths:\n            for seed in seeds:\n                print(f\'Joint generation: {count} glyphs in ONE image; strength={strength}; seed={seed}\', flush=True)\n                generated = pipeline(prompt=prompt, negative_prompt=negative, image=initial,\n                    strength=float(strength), num_inference_steps=steps, guidance_scale=7.5,\n                    generator=torch.Generator(\'cpu\').manual_seed(seed))\n                image = generated.images[0]\n                if image.size != initial.size:\n                    raise ValueError(\'Generator changed the sheet dimensions; fixed crops would be invalid\')\n                filtered = getattr(generated, \'nsfw_content_detected\', None)\n                save_page(image, slots, count, strength=float(strength), seed=seed,\n                          flags=bool(filtered[0]) if filtered is not None else None)\n    return report\n', 'diffusion_model/pilot_data.py': '"""Recover the transient stroke-training data for the diffusion scoring pilot."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom stroke_model.convert_reference_csv import existing_path\nfrom stroke_model.prepare_stroke_zips import prepare_stroke_zips\n\n\ndef _read_json(path):\n    return json.loads(path.read_text(encoding=\'utf-8\')) if path.is_file() else {}\n\n\ndef _usable_manifest(path):\n    if not path.is_file():\n        return False\n    rows = _read_json(path)\n    return (isinstance(rows, list) and bool(rows) and any(r.get(\'split\') == \'train\' for r in rows)\n            and all(((path.parent / row[\'path\']).is_file() if row.get(\'path\') else\n                     Path(row[\'reference\']).is_file() and all(Path(p).is_file() for p in row[\'reference_strokes\']))\n                    for row in rows))\n\n\ndef prepare_pilot_data(checkpoint, data_root=None, *, csv_path=None,\n                       local_root=\'/content/hsqm_stroke_zip\', manifest=None, reference_only=True):\n    """Default to reference-only preparation; retain full conversion as opt-in.\n\n    Reads the CSV snapshot, seed, resolution and supervision saved beside best.pt.\n    Does not train a model or modify checkpoint files. No whole-Drive search.\n    """\n    checkpoint = existing_path(checkpoint)\n    run = checkpoint.parent\n    config = _read_json(run/\'run_config.json\')\n    recorded = config.get(\'manifest\')\n    candidates = [Path(manifest)] if manifest is not None else ([Path(recorded)] if recorded else [])\n    for candidate in candidates:\n        if reference_only and manifest is None and candidate.is_file():\n            recorded_rows=_read_json(candidate)\n            if recorded_rows and recorded_rows[0].get(\'format\')!=\'reference_paths_v1\':\n                continue\n        if _usable_manifest(candidate):\n            print(\'현재 세션의 학습 데이터 재사용:\', candidate, flush=True)\n            return dict(manifest=str(candidate.resolve()), mode=\'reuse_live_training_data\',\n                        checkpoint=str(checkpoint.resolve()))\n    if manifest is not None:\n        print(\'지정한 학습 목록 또는 연결된 NPZ가 없어 원본 압축 자료로 다시 준비합니다.\', flush=True)\n    conversion = _read_json(run/\'conversion_report.json\')\n    if \'image_size\' not in config or \'seed\' not in config:\n        raise ValueError(\'체크포인트 옆 run_config.json에 image_size/seed가 필요합니다. \'\n                         \'기존 학습 결과 폴더의 best.pt를 선택하세요.\')\n    supervision = conversion.get(\'supervision\')\n    if supervision not in (\'actual_input_multihot\', \'exclusive_annotated\'):\n        raise ValueError(\'체크포인트 옆 conversion_report.json에서 기존 데이터 변환 방식을 확인할 수 없습니다.\')\n    # A saved CSV is the precise input to the original preparation, unlike a new\n    # CSV guessed from another dataset or a numeric character label.\n    snapshot = run/\'selected_paths.csv\'\n    if csv_path is not None:\n        selected_csv = existing_path(csv_path)\n        if snapshot.is_file() and selected_csv.read_bytes() != snapshot.read_bytes():\n            raise ValueError(\'CSV_PATH가 체크포인트의 학습 CSV와 다릅니다. 같은 CSV를 사용하세요.\')\n    elif snapshot.is_file():\n        selected_csv = snapshot\n    else:\n        raise FileNotFoundError(\'기존 학습 결과 폴더에 selected_paths.csv가 없습니다. \'\n                                \'CSV_PATH에 해당 학습에 쓴 원본 CSV를 지정하세요.\')\n\n    # Saved preparation metadata contains original archive paths. Infer their\n    # common dataset root only when the caller did not specify one.\n    if data_root is None:\n        import os\n        complete = _read_json(run/\'complete.json\')\n        archive_paths = [str(item[0]) for item in complete.get(\'archives\', [])]\n        if not archive_paths:\n            raise ValueError(\'DATA_ROOT에 기존 학습 ZIP/TAR가 있는 폴더를 지정하세요.\')\n        data_root = Path(os.path.commonpath(archive_paths))\n        if len(archive_paths) == 1:\n            data_root = data_root.parent\n    source = existing_path(data_root)\n    if reference_only:\n        from stroke_model.reference_only_data import prepare_reference_catalog\n        catalog=prepare_reference_catalog(source,selected_csv,conversion,int(config[\'image_size\']),\n                                          int(config[\'seed\']),local_root)\n        return dict(manifest=str(catalog.resolve()),mode=\'reference_only_no_training_npz\',\n            checkpoint=str(checkpoint.resolve()),data_root=str(source.resolve()),\n            csv_snapshot=str(selected_csv.resolve()),\n            csv_sha256=hashlib.sha256(selected_csv.read_bytes()).hexdigest(),\n            image_size=int(config[\'image_size\']),seed=int(config[\'seed\']),\n            supervision=supervision,original_image_content_fingerprint_available=False)\n    print(\'기존 임시 manifest 없음 → 학습 ZIP/TAR·CSV로 자동 재생성\', flush=True)\n    print(\'학습 압축 자료:\', source, flush=True)\n    print(\'원래 학습 CSV:\', selected_csv, flush=True)\n    print(\'기존 설정: size=\', config[\'image_size\'], \'seed=\', config[\'seed\'],\n          \'supervision=\', supervision, flush=True)\n    rebuilt = prepare_stroke_zips(source, local_root=local_root, csv_path=selected_csv,\n                                  size=int(config[\'image_size\']), seed=int(config[\'seed\']),\n                                  independent=supervision == \'actual_input_multihot\')\n    rebuilt_conversion = _read_json(rebuilt.parent/\'conversion_report.json\')\n    # Catch changed/missing source samples before claiming the restored split is\n    # the original training population. Image identity itself was not fingerprinted\n    # by old training runs and cannot be retroactively guaranteed.\n    for key in (\'supervision\', \'group_column\', \'skipped_count\'):\n        if key in conversion and conversion[key] != rebuilt_conversion.get(key):\n            raise ValueError(f\'재생성 데이터의 {key}가 원래 학습 기록과 다릅니다.\')\n    if \'rows\' in conversion and len(conversion[\'rows\']) != len(rebuilt_conversion.get(\'rows\', [])):\n        raise ValueError(\'재생성한 표본 수가 원래 학습 기록과 다릅니다.\')\n    if not _usable_manifest(rebuilt):\n        raise ValueError(\'재생성한 학습 목록/NPZ를 확인할 수 없습니다.\')\n    result = dict(manifest=str(rebuilt.resolve()), mode=\'rebuilt_from_training_archives\',\n                  checkpoint=str(checkpoint.resolve()), data_root=str(source.resolve()),\n                  csv_snapshot=str(selected_csv.resolve()),\n                  csv_sha256=hashlib.sha256(selected_csv.read_bytes()).hexdigest(),\n                  image_size=int(config[\'image_size\']), seed=int(config[\'seed\']),\n                  supervision=supervision, original_image_content_fingerprint_available=False)\n    print(\'학습 목록 준비 완료:\', rebuilt, flush=True)\n    return result\n', 'stroke_model/convert_reference_csv.py': '"""Strict legacy CSV adapter. Ambiguous/missing labels remain unknown (-1)."""\nimport ast\nimport hashlib\nimport json\nimport random\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\nfrom tqdm.auto import tqdm\n\n\ndef normalized(value):\n    return unicodedata.normalize(\'NFC\', str(value).replace(\'\\\\\', \'/\'))\n\n\ndef existing_path(value):\n    """Resolve NFC/NFD differences without assuming the shared drive spelling."""\n    path = Path(value)\n    if path.exists():\n        return path\n    current = Path(path.anchor) if path.is_absolute() else Path(\'.\')\n    parts = path.parts[1:] if path.is_absolute() else path.parts\n    for part in parts:\n        direct = current / part\n        if direct.exists():\n            current = direct\n            continue\n        if not current.is_dir():\n            raise FileNotFoundError(value)\n        matches = [p for p in current.iterdir() if normalized(p.name) == normalized(part)]\n        if len(matches) != 1:\n            raise FileNotFoundError(f\'경로를 찾거나 구분할 수 없습니다: {value}\')\n        current = matches[0]\n    return current\n\n\ndef path_list(value):\n    text = str(value).strip()\n    if not text or text.lower() in (\'nan\', \'none\'):\n        return []\n    try:\n        parsed = ast.literal_eval(text)\n        if isinstance(parsed, (list, tuple)):\n            return [str(x) for x in parsed]\n    except (ValueError, SyntaxError):\n        pass\n    for sep in (\';\', \'|\', \'\\n\', \',\'):\n        if sep in text:\n            return [p.strip().strip(\'\\"\\\'\') for p in text.split(sep) if p.strip()]\n    return [text.strip(\'\\"\\\'\')]\n\n\nclass Resolver:\n    def __init__(self, csv_path, dataset_root, local_only=False):\n        self.csv_path, self.root = csv_path, dataset_root\n        self.local_only = local_only\n        self.index = None\n\n    def __call__(self, value):\n        raw = normalized(str(value).strip().strip(\'\\"\\\'\'))\n        candidates = (Path(raw), self.csv_path.parent / raw, self.root / raw)\n        if self.local_only:\n            candidate = (self.root / raw).resolve()\n            candidates = (candidate,) if candidate.is_relative_to(self.root.resolve()) else ()\n        for candidate in candidates:\n            try:\n                found = existing_path(candidate)\n                if found.is_file():\n                    return found\n            except FileNotFoundError:\n                pass\n        if self.index is None:\n            self.index = {}\n            for p in self.root.rglob(\'*\'):\n                if \'input_stroke_ga\' in p.relative_to(self.root).parts:\n                    continue  # Legacy config excludes this duplicate, unused folder.\n                if p.is_file() and p.suffix.lower() in (\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'):\n                    self.index.setdefault(normalized(p.name), []).append(p)\n        candidates = self.index.get(raw.split(\'/\')[-1], [])\n        if not candidates:\n            raise FileNotFoundError(f\'CSV 이미지가 없습니다: {value} (검색 범위: {self.root})\')\n        def suffix_score(p):\n            score = 0\n            for left, right in zip(reversed(normalized(p).split(\'/\')), reversed(raw.split(\'/\'))):\n                if left != right:\n                    break\n                score += 1\n            return score\n        best = max(map(suffix_score, candidates))\n        matches = [p for p in candidates if suffix_score(p) == best]\n        if len(matches) != 1:\n            raise ValueError(f\'같은 파일명이 여러 곳에 있습니다. CSV 경로를 명확하게 해주세요: {value}: {matches}\')\n        return matches[0]\n\n\ndef load_ink(path, size):\n    with Image.open(path) as source:\n        # Composite transparent PNGs onto white before converting to grayscale.\n        image = ImageOps.exif_transpose(source).convert(\'RGBA\')\n        background = Image.new(\'RGBA\', image.size, (255, 255, 255, 255))\n        image = Image.alpha_composite(background, image).convert(\'L\')\n        x = np.asarray(image.resize((size, size), Image.Resampling.BILINEAR), dtype=np.float32) / 255\n    if x.mean() > .5:\n        x = 1 - x\n    return x[None].copy()\n\n\ndef convert_csv(csv_path, output, size=128, seed=42, dataset_root=None, split=None, local_only=False,\n                independent=False):\n    if split not in (None, \'test\'):\n        raise ValueError(\'split must be None (train/val) or test\')\n    csv_path = existing_path(csv_path)\n    root = existing_path(dataset_root) if dataset_root else csv_path.parent\n    resolve = Resolver(csv_path, root, local_only=local_only)\n    df = pd.read_csv(csv_path)\n    names = {str(c).lower(): c for c in df.columns}\n    def column(*aliases, optional=False):\n        for name in aliases:\n            if name.lower() in names:\n                return names[name.lower()]\n        if optional:\n            return None\n        raise ValueError(f\'필수 컬럼 {aliases} 없음. 현재 CSV 컬럼: {list(df.columns)}\')\n    cols = {\n        \'image\': column(\'I_prep\', \'input_image_path\', \'input_path\', \'input_image\', \'input\', \'source_path\'),\n        \'reference\': column(\'I_g\', \'reference_image\', \'reference_path\', \'target_image\', \'target_path\', \'glyph_path\'),\n        \'strokes\': column(\'I_G_strokes\', \'reference_strokes\', \'reference_stroke_paths\', \'template_strokes\', \'target_strokes\', \'target_stroke_paths\'),\n        \'targets\': column(\'input_stroke_paths\', \'input_strokes\', \'input_strokes_paths\', \'prep_stroke_paths\', \'gt_stroke_paths\', \'gt_strokes\', \'answer_stroke_paths\', \'stroke_paths\'),\n    }\n    group_col = column(\'writer_id\', \'writer\', \'subject_id\', \'group_id\', optional=True)\n    char_col = column(\'char_id\', \'char\', \'character\', \'text\', optional=True)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    (output / \'samples\').mkdir()\n    rows, reports, skipped_rows = [], [], []\n    seen_images = {}\n    for index, row in tqdm(df.iterrows(), total=len(df), desc=\'Convert images to NPZ\', unit=\'sample\'):\n        try:\n            image_path = resolve(row[cols[\'image\']])\n            image = load_ink(image_path, size)\n            reference = load_ink(resolve(row[cols[\'reference\']]), size)\n            ref_paths, target_paths = path_list(row[cols[\'strokes\']]), path_list(row[cols[\'targets\']])\n            if not ref_paths or len(ref_paths) != len(target_paths):\n                raise ValueError(f\'기준/정답 획 수 불일치: {len(ref_paths)} / {len(target_paths)}\')\n            strokes = np.concatenate([load_ink(resolve(p), size) for p in ref_paths])\n            targets = np.concatenate([load_ink(resolve(p), size) for p in target_paths]) > .5\n            empty_reference = strokes.reshape(len(strokes), -1).max(1) <= .5\n            if empty_reference.any():\n                empty_paths = [p for p, empty in zip(ref_paths, empty_reference) if empty]\n                if independent:\n                    skipped_rows.append({\'csv_row\': int(index) + 1,\n                        \'input_path\': str(row[cols[\'image\']]),\n                        \'empty_reference_paths\': \';\'.join(empty_paths),\n                        \'reason\': \'empty_reference_stroke_after_resize\'})\n                    if len(skipped_rows) <= 5:\n                        print(f\'건너뜀: CSV {index + 1}번째 행, 빈 기준 획: {empty_paths}\', flush=True)\n                    continue\n                raise ValueError(f\'비어 있는 기준 획 이미지가 있습니다: {empty_paths}\')\n            foreground = image[0] > .5\n            counts = targets.sum(0)\n            known = foreground & (counts == 1)\n            labels = np.full((size, size), -1, dtype=np.int64)\n            labels[known] = targets.argmax(0)[known]\n            if not independent and any(not (labels == k).any() for k in range(len(strokes))):\n                raise ValueError(\'정답 획 중 단독 소유 픽셀이 없는 획이 있습니다. 획 마스크/극성을 확인해주세요\')\n            digest = hashlib.sha256(image.tobytes()).hexdigest()\n            if group_col:\n                if pd.isna(row[group_col]):\n                    raise ValueError(\'필자/그룹 ID가 비어 있습니다\')\n                group = \'writer:\' + str(row[group_col])\n            elif independent:\n                # Keep augmented variants sharing a font/character reference together.\n                group = \'reference:\' + hashlib.sha256(reference.tobytes()).hexdigest()\n            else:\n                group = \'image:\' + digest\n            if digest in seen_images and seen_images[digest] != group:\n                raise ValueError(\'동일 입력 이미지에 서로 다른 필자/그룹 ID가 지정되었습니다\')\n            seen_images[digest] = group\n            path = output / \'samples\' / f\'{index:05d}.npz\'\n            arrays = dict(image=image, reference=reference, strokes=strokes, labels=labels)\n            if independent:\n                # Actual input annotations, including shared crossings and empty slots.\n                # These are NOT reconstructed from exclusive labels or templates.\n                arrays.update(target_masks=targets.astype(np.float32),\n                              valid_pixels=np.ones_like(image))\n            np.savez_compressed(path, **arrays)\n            coverage = float(known.sum() / max(foreground.sum(), 1))\n            rows.append({\'path\': str(path.resolve()), \'group_id\': group, \'source\': \'legacy_csv_unreviewed\',\n                         \'char_id\': str(row[char_col]) if char_col else \'\', \'source_image\': str(image_path)})\n            reports.append({\'row\': int(index), \'foreground_pixels\': int(foreground.sum()),\n                            \'annotation_coverage\': coverage, \'overlap_pixels\': int((foreground & (counts > 1)).sum()),\n                            \'missing_pixels\': int((foreground & (counts == 0)).sum())})\n        except (ValueError, FileNotFoundError) as exc:\n            raise ValueError(f\'CSV 데이터 {index + 1}번째 행: {exc}\') from exc\n    pd.DataFrame(skipped_rows, columns=[\'csv_row\', \'input_path\', \'empty_reference_paths\', \'reason\']).to_csv(\n        output / \'skipped_samples.csv\', index=False, encoding=\'utf-8-sig\')\n    groups = sorted({r[\'group_id\'] for r in rows})\n    if not rows:\n        raise ValueError(\'CSV에 데이터가 없습니다\')\n    if split is None and len(groups) < 2:\n        raise ValueError(\'검증 분리를 위해 최소 2개 필자/원본 그룹이 필요합니다\')\n    random.Random(seed).shuffle(groups)\n    validation = set(groups[:max(1, round(len(groups) * .2))])\n    for row in rows:\n        row[\'split\'] = split or (\'val\' if row[\'group_id\'] in validation else \'train\')\n    manifest = output / \'manifest.json\'\n    manifest.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    report = {\'csv\': str(csv_path), \'columns\': cols, \'group_column\': group_col,\n              \'split_note\': \'전체 test 전용, 학습/증강 없음\' if split == \'test\' else (\n                  \'필자 그룹 분할\' if group_col else (\n                      \'기준 이미지 그룹 분할: 같은 기준의 합성 변형은 같은 split; 필자 일반화 평가는 아님\'\n                      if independent else \'원본 이미지 분할: 필자 일반화 평가는 아님\')),\n              \'label_policy\': (\'실제 입력 획별 이진 마스크 전체 감독; 교차점 공유/빈 획 보존. CSV 획 순서 대응 가정.\'\n                               if independent else \'단독 소유 전경만 감독; 교차/미할당은 -1. CSV 획 순서가 대응한다고 가정.\'),\n              \'supervision\': \'actual_input_multihot\' if independent else \'exclusive_annotated\',\n              \'skipped_count\': len(skipped_rows), \'skipped_rows\': skipped_rows,\n              \'rows\': reports}\n    (output / \'conversion_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(f"변환 완료: train={sum(r[\'split\']==\'train\' for r in rows)}, val={sum(r[\'split\']==\'val\' for r in rows)}, test={sum(r[\'split\']==\'test\' for r in rows)}")\n    print(report[\'split_note\'])\n    print(report[\'label_policy\'])\n    print(f\'빈 기준 획으로 제외한 샘플: {len(skipped_rows)}개 (skipped_samples.csv)\')\n    print(\'획 순서의 의미적 대응은 자동 검증되지 않습니다. conversion_report.json과 미리보기를 확인하세요.\')\n    return manifest\n', 'stroke_model/prepare_stroke_zips.py': '"""Stage the four stroke archives locally; use CSV for semantic correspondence."""\nimport hashlib\nimport json\nimport shutil\nimport stat\nimport tarfile\nimport tempfile\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nfrom tqdm.auto import tqdm\n\nfrom .convert_reference_csv import convert_csv, existing_path\n\nARCHIVES = (\'input_stroke.zip\', \'input.zip\', \'target_stroke.zip\', \'target.zip\')\n\n\ndef discover_archives(source, roles=ARCHIVES):\n    """Discover every supported archive recursively, regardless of name/count."""\n    paths = []\n    for name in roles:\n        role = Path(name).stem\n        try:\n            folder = existing_path(source / role)\n        except FileNotFoundError:\n            folder = source / role\n        parts = []\n        if folder.is_dir():\n            extensions = (\'.tar\', \'.tar.gz\', \'.tgz\', \'.tar.bz2\', \'.tbz2\', \'.tar.xz\', \'.txz\', \'.zip\')\n            parts = sorted((path for path in folder.rglob(\'*\')\n                            if path.is_file() and path.name.casefold().endswith(extensions)),\n                           key=lambda path: path.relative_to(folder).as_posix().casefold())\n        if parts:\n            paths.extend(parts)\n        else:\n            try:\n                paths.append(existing_path(source / name))\n            except FileNotFoundError as exc:\n                raise FileNotFoundError(\n                    f\'{source / role} 안의 TAR/ZIP 압축 파일 \'\n                    f\'또는 {source / name}을 찾을 수 없습니다.\') from exc\n    return paths\n\n\ndef extract_checked(archive, destination):\n    destination = Path(destination).resolve()\n    if not zipfile.is_zipfile(archive):\n        with tarfile.open(archive, \'r:*\') as tar:\n            members = tar.getmembers()\n            for member in members:\n                path = PurePosixPath(member.name.replace(\'\\\\\', \'/\'))\n                if (path.is_absolute() or \'..\' in path.parts or\n                        any(\':\' in part for part in path.parts) or\n                        not (member.isfile() or member.isdir())):\n                    raise ValueError(f\'Unsafe TAR member: {member.name}\')\n            needed = sum(m.size for m in members if m.isfile())\n            if needed > shutil.disk_usage(destination.parent).free:\n                raise OSError(f\'로컬 디스크 부족: 압축 해제에 {needed / 2**30:.1f} GiB 필요\')\n            for member in tqdm(members, desc=f\'Extract {Path(archive).name}\', unit=\'file\', mininterval=2):\n                target = destination.joinpath(*PurePosixPath(member.name.replace(\'\\\\\', \'/\')).parts)\n                if member.isdir():\n                    target.mkdir(parents=True, exist_ok=True)\n                else:\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with tar.extractfile(member) as src, target.open(\'xb\') as dst:\n                        shutil.copyfileobj(src, dst)\n        return\n    with zipfile.ZipFile(archive) as z:\n        members = z.infolist()\n        for member in members:\n            path = PurePosixPath(member.filename.replace(\'\\\\\', \'/\'))\n            if (path.is_absolute() or \'..\' in path.parts or\n                    any(\':\' in part for part in path.parts) or\n                    stat.S_ISLNK(member.external_attr >> 16)):\n                raise ValueError(f\'Unsafe ZIP member: {member.filename}\')\n        needed = sum(m.file_size for m in members)\n        if needed > shutil.disk_usage(destination.parent).free:\n            raise OSError(f\'로컬 디스크 부족: 압축 해제에 {needed / 2**30:.1f} GiB 필요\')\n        for member in tqdm(members, desc=f\'Extract {Path(archive).name}\', unit=\'file\', mininterval=2):\n            z.extract(member, destination)\n\n\ndef find_csv(source, raw, csv_path=None):\n    if csv_path:\n        return existing_path(csv_path)\n    # Only the selected dataset directory and extracted archives, never a test/old dataset.\n    direct = [p for p in source.iterdir() if p.is_file() and p.name.casefold() in (\'path.csv\', \'paths.csv\')]\n    candidates = direct or [p for p in raw.rglob(\'*\') if p.is_file() and p.name.casefold() in (\'path.csv\', \'paths.csv\')]\n    if not candidates:\n        raise FileNotFoundError(\n            f\'path.csv / paths.csv가 아직 없습니다. {source}에 업로드한 뒤 이 셀만 다시 실행하세요. \'\n            \'완료된 압축 파일 복사/해제는 재사용합니다. 다른 CSV 이름이면 CSV_PATH를 지정하세요.\')\n    if len(candidates) != 1:\n        raise ValueError(f\'CSV 후보가 여러 개입니다. CSV_PATH를 지정하세요: {candidates}\')\n    return candidates[0]\n\n\ndef stage_stroke_archives(source, local_root=\'/content/hsqm_stroke_zip\', roles=None):\n    """Share the training archive cache while staging only requested roles."""\n    source = existing_path(source)\n    paths = discover_archives(source)\n    signature = [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in paths]\n    key = hashlib.sha256(b\'archive_layout2:\' + json.dumps(signature).encode()).hexdigest()[:16]\n    cache = Path(local_root) / key\n    raw = cache / \'raw\'\n    raw.mkdir(parents=True, exist_ok=True)\n    allowed = set(discover_archives(source, roles=roles)) if roles is not None else set(paths)\n    print(\'사용할 압축 파일:\',len(allowed),\'개 / 전체\',len(paths),\'개\',flush=True)\n    for path in paths:\n        if path not in allowed:\n            continue\n        # Different folders may contain archives with identical filenames.\n        archive_id = hashlib.sha256(path.relative_to(source).as_posix().encode()).hexdigest()[:16]\n        destination = raw / archive_id\n        if (destination / \'.complete.json\').is_file():\n            print(\'로컬 압축 데이터 재사용:\', path.name, flush=True)\n            continue\n        # Failed copies/extractions never create the completion marker.\n        local_zip = cache / (archive_id + \'_\' + path.name)\n        before = (path.stat().st_size, path.stat().st_mtime_ns)\n        if shutil.disk_usage(cache).free < before[0]:\n            raise OSError(\'압축 파일 복사를 위한 로컬 디스크 공간이 부족합니다\')\n        print(\'Drive → 로컬 압축 파일 복사:\', path.name, flush=True)\n        partial = local_zip.with_suffix(\'.partial\')\n        with path.open(\'rb\') as src, partial.open(\'wb\') as dst:\n            with tqdm(total=before[0], desc=path.name, unit=\'B\', unit_scale=True, mininterval=2) as progress:\n                while chunk := src.read(8 * 1024 * 1024):\n                    dst.write(chunk)\n                    progress.update(len(chunk))\n        after = (path.stat().st_size, path.stat().st_mtime_ns)\n        if before != after or partial.stat().st_size != before[0]:\n            raise RuntimeError(f\'{path.name}이 복사 중 변경됐습니다. 업로드 완료 후 재실행하세요.\')\n        partial.replace(local_zip)\n        temporary = Path(tempfile.mkdtemp(prefix=path.stem + \'_\', dir=cache))\n        extract_checked(local_zip, temporary)\n        (temporary / \'.complete.json\').write_text(json.dumps(before), encoding=\'utf-8\')\n        temporary.rename(destination)\n        local_zip.unlink()  # Keep extracted images, reclaim the local archive space.\n    return cache, raw, signature\n\n\ndef prepare_stroke_zips(source, local_root=\'/content/hsqm_stroke_zip\', csv_path=None, size=128, seed=42,\n                        independent=False):\n    source = existing_path(source)\n    cache, raw, signature = stage_stroke_archives(source, local_root)\n    selected = find_csv(source, raw, csv_path)\n    content = selected.read_bytes()\n    conversion_key = hashlib.sha256(content + f\'converter3:{size}:{seed}:{independent}\'.encode()).hexdigest()[:16]\n    prepared = cache / (\'prepared_\' + conversion_key)\n    manifest = prepared / \'manifest.json\'\n    if (prepared / \'.complete.json\').is_file() and manifest.is_file():\n        print(\'로컬 변환 데이터 재사용:\', manifest, flush=True)\n        return manifest\n    print(\'CSV 선택:\', selected, flush=True)\n    local_csv = cache / \'selected_paths.csv\'\n    local_csv.write_bytes(content)\n    temporary = Path(tempfile.mkdtemp(prefix=\'conversion_\', dir=cache)) / \'prepared\'\n    manifest = convert_csv(local_csv, temporary, size=size, seed=seed, dataset_root=raw, local_only=True,\n                           independent=independent)\n    # Keep converter absolute NPZ paths stable: use the completed temp directory directly.\n    prepared.mkdir(exist_ok=True)\n    rows = json.loads(manifest.read_text(encoding=\'utf-8\'))\n    (prepared / \'manifest.json\').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    shutil.copyfile(temporary / \'conversion_report.json\', prepared / \'conversion_report.json\')\n    shutil.copyfile(temporary / \'skipped_samples.csv\', prepared / \'skipped_samples.csv\')\n    (prepared / \'selected_paths.csv\').write_bytes(content)\n    (prepared / \'.complete.json\').write_text(json.dumps({\'csv_source\': str(selected), \'archives\': signature}), encoding=\'utf-8\')\n    return prepared / \'manifest.json\'\n', 'stroke_model/reference_only_data.py': '"""Restore training-reference membership without converting training samples."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\n\nimport numpy as np\nimport pandas as pd\nfrom tqdm.auto import tqdm\n\nfrom .convert_reference_csv import Resolver, load_ink, normalized, path_list\nfrom .prepare_stroke_zips import stage_stroke_archives, discover_archives\n\nREFERENCE_ROLES = (\'target.zip\', \'target_stroke.zip\')\n\n\ndef prepare_reference_catalog(source, csv_path, conversion, size, seed, local_root):\n    """Reproduce the original group split from the saved accepted CSV row IDs.\n\n    Only reference PNGs are decoded to recover content-hash reference groups.\n    Ordered stroke PNGs are opened later, when a glyph is selected. No NPZ here.\n    """\n    source, csv_path = Path(source), Path(csv_path)\n    group_col = conversion.get(\'group_column\')\n    if conversion.get(\'supervision\') != \'actual_input_multihot\' and not group_col:\n        raise ValueError(\'이전 image-hash 분할은 기준만으로 복원할 수 없습니다. \'\n                         \'REFERENCE_ONLY=False로 기존 전체 변환을 사용하세요.\')\n    if \'rows\' not in conversion:\n        raise ValueError(\'저장된 conversion_report.json에 원래 학습 행 목록이 필요합니다.\')\n    cache, raw, _ = stage_stroke_archives(source, local_root, roles=REFERENCE_ROLES)\n    fingerprint = hashlib.sha256(csv_path.read_bytes()+json.dumps(\n        dict(version=1,size=size,seed=seed,conversion=conversion),sort_keys=True).encode()).hexdigest()[:20]\n    destination = cache/f\'reference_only_{fingerprint}\'\n    manifest = destination/\'manifest.json\'\n    if manifest.is_file():\n        rows=json.loads(manifest.read_text(encoding=\'utf-8\'))\n        if rows and all(Path(r[\'reference\']).is_file() and all(Path(p).is_file() for p in r[\'reference_strokes\']) for r in rows):\n            print(\'기준 글자 목록 캐시 재사용 (학습 NPZ 변환 없음):\',manifest,flush=True)\n            return manifest\n    frame=pd.read_csv(csv_path)\n    columns=conversion[\'columns\']\n    accepted=[int(row[\'row\']) for row in conversion[\'rows\']]\n    if not accepted or len(set(accepted))!=len(accepted) or min(accepted)<0 or max(accepted)>=len(frame):\n        raise ValueError(\'저장된 학습 행과 CSV가 일치하지 않습니다.\')\n    resolve=Resolver(csv_path,raw,local_only=True)\n    # Index ONLY target archives, even if input archives were extracted earlier.\n    resolve.index={}\n    for archive in discover_archives(source,roles=REFERENCE_ROLES):\n        archive_id=hashlib.sha256(archive.relative_to(source).as_posix().encode()).hexdigest()[:16]\n        for path in (raw/archive_id).rglob(\'*\'):\n            if path.is_file() and path.suffix.lower() in (\'.png\',\'.jpg\',\'.jpeg\',\'.bmp\',\'.webp\',\'.tif\',\'.tiff\'):\n                resolve.index.setdefault(normalized(path.name),[]).append(path)\n    char_col=next((c for c in frame.columns if str(c).lower() in (\'char_id\',\'char\',\'character\',\'text\')),None)\n    hashes, resolved, records, groups = {}, {}, [], set()\n    seen=set()\n    print(\'학습 입력/정답 획은 읽지 않고 기준 글자만 준비합니다.\',flush=True)\n    for index in tqdm(accepted,desc=\'Index reference glyphs (no NPZ)\',unit=\'row\',mininterval=2):\n        row=frame.iloc[index]\n        raw_reference=str(row[columns[\'reference\']])\n        if raw_reference not in resolved:\n            resolved[raw_reference]=resolve(raw_reference)\n        reference=resolved[raw_reference]\n        if str(reference) not in hashes:\n            hashes[str(reference)]=hashlib.sha256(load_ink(reference,size).tobytes()).hexdigest()\n        if group_col:\n            if pd.isna(row[group_col]):raise ValueError(\'학습 그룹 ID가 비어 있습니다.\')\n            group=\'writer:\'+str(row[group_col])\n        else:\n            group=\'reference:\'+hashes[str(reference)]\n        groups.add(group)\n        stroke_names=path_list(row[columns[\'strokes\']])\n        if not stroke_names:raise ValueError(\'기준 획 목록이 비어 있습니다.\')\n        key=(str(reference),tuple(stroke_names),group)\n        if key in seen:continue\n        seen.add(key)\n        paths=[]\n        for name in stroke_names:\n            if name not in resolved:resolved[name]=resolve(name)\n            paths.append(str(resolved[name].resolve()))\n        signature=hashlib.sha256((hashes[str(reference)]+json.dumps(paths)).encode()).hexdigest()\n        records.append(dict(reference=str(reference.resolve()),reference_strokes=paths,\n            image_size=size,group_id=group,char_id=str(row[char_col]) if char_col else \'\',\n            source_csv_row=index+1,reference_sha256=signature,format=\'reference_paths_v1\'))\n    groups=sorted(groups)\n    if len(groups)<2:raise ValueError(\'원래 학습 분할을 복원할 그룹이 부족합니다.\')\n    random.Random(seed).shuffle(groups)\n    validation=set(groups[:max(1,round(len(groups)*.2))])\n    for record in records:record[\'split\']=\'val\' if record[\'group_id\'] in validation else \'train\'\n    destination.mkdir(exist_ok=True)\n    partial=manifest.with_suffix(\'.partial\')\n    partial.write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    partial.replace(manifest)\n    print(f\'기준 {len(records)}개 준비 완료. 학습 표본 {len(accepted)}개의 NPZ 변환을 생략했습니다.\',flush=True)\n    return manifest\n'}

ROOT=Path('/content/hsqm_previous_adaptive');ROOT.mkdir(exist_ok=True)
for package in ('diffusion_model','stroke_model'):
    (ROOT/package).mkdir(exist_ok=True);(ROOT/package/'__init__.py').touch()
for name,code in RUNTIME_HELPERS.items():(ROOT/name).write_text(code,encoding='utf-8')
for name,expected in PINNED_HASHES.items():
    actual=hashlib.sha256((ROOT/name).read_bytes()).hexdigest()
    if actual!=expected:raise RuntimeError('고정 코드 해시 불일치: '+name)
sys.path.insert(0,str(ROOT))
for name in list(sys.modules):
    if name.split('.')[0] in ('diffusion_model','stroke_model'):sys.modules.pop(name)
import torch,numpy as np,matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display,clear_output
from google.colab import drive,files
from stroke_model.convert_reference_csv import existing_path
from stroke_model.adaptive_ink_constraints import checkpoint_hash
from stroke_model.reference_model import ReferenceStrokeModel
from diffusion_model.pilot_data import prepare_pilot_data
from diffusion_model.score_pilot import training_catalog,load_catalog_reference
from diffusion_model.selectable_adaptive_sweep import select_entries,prepare_groups,generate_cases,repair_cases,validate_generation_cases
if not torch.cuda.is_available():raise RuntimeError('런타임에서 L4 GPU를 선택하세요.')
print('GPU:',torch.cuda.get_device_name(0),'| 보정 버전:',ADAPTIVE_COMMIT)

In [ ]:
# 기존 코랩 세션의 manifest가 남아 있으면 MANIFEST에 직접 지정해도 됩니다.
CHECKPOINT=''
MANIFEST=''
DATA_ROOT=''  # 비우면 학습 폴더의 complete.json에서 기존 압축 자료 위치 복구
CSV_PATH=''   # 비우면 체크포인트 옆 selected_paths.csv 사용
USE_DRIVE=True
EXPECTED_SHA='92b8d5dfac59a4e44d632464836877e1cc242b2a4bc72cd6e58d5fcef84cbbf7'
if USE_DRIVE and not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
if not CHECKPOINT:
    recorded='/content/drive/Shareddrives/2026 자율연구/HSQM/v2_topology_runs/20261104_214705_208224/best.pt'
    try:
        candidate=existing_path(recorded)
        if checkpoint_hash(candidate)==EXPECTED_SHA:CHECKPOINT=str(candidate)
    except FileNotFoundError:pass
if not CHECKPOINT:
    for root in (Path('/content/drive/Shareddrives'),Path('/content/drive/MyDrive')):
        if not root.is_dir():continue
        print('기존 학습 체크포인트 검색:',root,flush=True)
        for path in root.rglob('best.pt'):
            try:matched=checkpoint_hash(path)==EXPECTED_SHA
            except OSError:continue
            if matched:CHECKPOINT=str(path);break
        if CHECKPOINT:break
if not CHECKPOINT:raise FileNotFoundError('CHECKPOINT에 기존 학습 폴더의 best.pt 경로를 지정하세요.')
if checkpoint_hash(CHECKPOINT)!=EXPECTED_SHA:raise ValueError('이전 adaptive에서 쓴 체크포인트와 다릅니다.')
preparation=prepare_pilot_data(CHECKPOINT,data_root=DATA_ROOT or None,csv_path=CSV_PATH or None,
    manifest=MANIFEST or None,reference_only=True)
MANIFEST=preparation['manifest'];CATALOG=training_catalog(MANIFEST)
if len(CATALOG)<4:raise ValueError('선택 가능한 서로 다른 기준 글자가 4개 미만입니다.')
state=torch.load(CHECKPOINT,map_location='cpu',weights_only=True)
MODEL_SIZE=state['image_size']
if state.get('architecture')!='reference_v2_topology_loss_v1':raise ValueError('기존 V2 topology 체크포인트가 필요합니다.')
extractor=ReferenceStrokeModel(image_size=state['image_size'],width=state['width'],pretrained=False)
extractor.load_state_dict(state['model'],strict=True);extractor=extractor.eval().requires_grad_(False)
del state
print('글자 목록:',len(CATALOG),'개 | 모델:',CHECKPOINT)
print('다음 셀에서 그림을 보고 네 글자를 선택하세요.')

In [ ]:
import ipywidgets as widgets
OPTIONS=[(f"row {r['manifest_row']} | {r.get('char_id','') or '그림으로 확인'}",r['manifest_row']) for r in CATALOG]
by_row={r['manifest_row']:r for r in CATALOG}
selectors=[widgets.Dropdown(options=OPTIONS,value=CATALOG[i]['manifest_row'],description=f'칸 {i+1}',
           layout=widgets.Layout(width='95%')) for i in range(4)]
page=widgets.IntSlider(value=0,min=0,max=(len(CATALOG)-1)//16,description='목록 페이지',continuous_update=False)
gallery=widgets.Output();preview=widgets.Output();group_output=widgets.Output()
SELECTED_GROUPS=[]
def draw_gallery(change=None):
    with gallery:
        clear_output(wait=True)
        shown=CATALOG[page.value*16:(page.value+1)*16]
        fig,axes=plt.subplots(4,4,figsize=(10,10))
        for ax in axes.flat:ax.axis('off')
        for ax,entry in zip(axes.flat,shown):
            reference,_=load_catalog_reference(entry)
            ax.imshow(1-reference[0],cmap='gray',vmin=0,vmax=1)
            ax.set_title(f"row {entry['manifest_row']} | {entry.get('char_id','')}")
        fig.tight_layout();plt.show();plt.close(fig)
def draw_selected(change=None):
    with preview:
        clear_output(wait=True)
        rows=[s.value for s in selectors]
        print('현재 네 칸 순서:',rows)
        fig,axes=plt.subplots(2,2,figsize=(6,6))
        for ax,row in zip(axes.flat,rows):
            reference,_=load_catalog_reference(by_row[row])
            ax.imshow(1-reference[0],cmap='gray',vmin=0,vmax=1);ax.set_title(f'row {row}');ax.axis('off')
        plt.show();plt.close(fig)
def add_group(button):
    with group_output:
        clear_output(wait=True)
        rows=tuple(s.value for s in selectors)
        try:select_entries(MANIFEST,rows)
        except ValueError as exc:print(exc);return
        if rows not in SELECTED_GROUPS:SELECTED_GROUPS.append(rows)
        print('등록된 조합:',SELECTED_GROUPS)
def clear_groups(button):
    SELECTED_GROUPS.clear()
    with group_output:clear_output();print('등록 해제. 4번을 실행하면 현재 네 칸을 사용합니다.')
add=widgets.Button(description='현재 4자 조합 추가',button_style='primary');add.on_click(add_group)
reset=widgets.Button(description='조합 목록 초기화');reset.on_click(clear_groups)
page.observe(draw_gallery,names='value')
for selector in selectors:selector.observe(draw_selected,names='value')
display(page,gallery,widgets.VBox(selectors),preview,widgets.HBox([add,reset]),group_output)
draw_gallery();draw_selected()
print('한 조합만 시험하려면 네 칸을 고른 뒤 바로 4번 실행. 여러 조합은 추가 버튼으로 등록하세요.')

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline,DDIMScheduler
GENERATION_STRENGTH=.65
GENERATION_SEED=100
GENERATION_STEPS=100
MODEL_ID='stable-diffusion-v1-5/stable-diffusion-v1-5'
REVISION='451f4fe16113bff5a5d2269ed5ad43b0592e9a14'  # 이전 실험과 같은 확산 모델 revision
OUTPUT=Path('/content')/('selectable_adaptive_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
selections=list(SELECTED_GROUPS) or [tuple(s.value for s in selectors)]
GROUPS=prepare_groups(MANIFEST,selections,OUTPUT)
for group in GROUPS:
    for entry in training_catalog(group['manifest']):
        ref,strokes=load_catalog_reference(entry)
        if ref.shape[-2:]!=(MODEL_SIZE,MODEL_SIZE):raise ValueError('선택한 기준 글자와 모델 해상도가 다릅니다.')
print('실제 생성 조합:',selections,'| 강도:',GENERATION_STRENGTH,'| seed:',GENERATION_SEED)
extractor.cpu();gc.collect();torch.cuda.empty_cache()
generation_pipe=StableDiffusionImg2ImgPipeline.from_pretrained(MODEL_ID,revision=REVISION,
    torch_dtype=torch.float16,use_safetensors=True)
generation_pipe.scheduler=DDIMScheduler.from_config(generation_pipe.scheduler.config)
generation_pipe.enable_attention_slicing();generation_pipe.to('cuda')
try:
    CASES=generate_cases(GROUPS,strengths=(GENERATION_STRENGTH,),seeds=(GENERATION_SEED,),steps=GENERATION_STEPS,
        revision=REVISION,pipeline=generation_pipe,include_control=False)
finally:
    del generation_pipe;gc.collect();torch.cuda.empty_cache()
for group in GROUPS:
    failure=Path(group['root'])/'generation_failure.json'
    if failure.is_file():print('생성 중 실패 기록:',failure.read_text())
(OUTPUT/'cases.json').write_text(json.dumps(CASES,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT/'experiment.json').write_text(json.dumps(dict(adaptive_commit=ADAPTIVE_COMMIT,
    pinned_source_hashes=PINNED_HASHES,checkpoint_sha256=EXPECTED_SHA,selections=selections,
    generation_strength=GENERATION_STRENGTH,generation_seed=GENERATION_SEED,generation_steps=GENERATION_STEPS,
    model_id=MODEL_ID,revision=REVISION,free_text_generation=False,
    reference_initialized_generation=True,data_preparation=preparation),ensure_ascii=False,indent=2),encoding='utf-8')
if not CASES:raise RuntimeError('생성된 사례가 없습니다. generation_failure.json을 확인하세요.')
fig,axes=plt.subplots((len(CASES)+2)//3,3,figsize=(15,5*((len(CASES)+2)//3)),squeeze=False)
for ax in axes.flat:ax.axis('off')
for ax,case in zip(axes.flat,CASES):
    ax.imshow(Image.open(case['input']));ax.set_title(case['case_id']+'\n'+case['label'])
fig.tight_layout();fig.savefig(OUTPUT/'generation_gallery.png',dpi=100);plt.show();plt.close(fig)
print('생성 사례:',len(CASES),'개 — 각 글자 조합당 한 장만 보정합니다. 다른 seed는 이 셀부터 다시 실행하세요.')

In [ ]:
# None이면 모든 사례. 특정 경우만 시험하려면 위에 표시된 case_id 목록을 지정하세요.
REPAIR_CASE_IDS=None
REPAIR_STEPS=60
REPAIR_STRENGTH=.25
REPAIR_SEED=2026
validate_generation_cases(CASES,seed=GENERATION_SEED,strength=GENERATION_STRENGTH)
selected_cases=CASES if REPAIR_CASE_IDS is None else [c for c in CASES if c['case_id'] in REPAIR_CASE_IDS]
if REPAIR_CASE_IDS is not None and set(REPAIR_CASE_IDS)!={c['case_id'] for c in selected_cases}:
    raise ValueError('REPAIR_CASE_IDS에 없는 case_id가 있습니다.')
if not selected_cases:raise ValueError('보정할 사례가 없습니다.')
attempt='attempt_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f')
selected_cases=[dict(c,output=str(Path(c['output'])/attempt)) for c in selected_cases]
extractor.to('cuda')
def progress(method,step,total,row):
    if step==1 or step%5==0 or step==total:print(method,step,'/',total,'accepted=',row['accepted_inner_steps'],flush=True)
def show_case(row):
    print(row['case_id'],row['status'],flush=True)
    if row['status']!='completed':print(row.get('error'));return
    fig,axes=plt.subplots(1,3,figsize=(15,5))
    for ax,(key,title) in zip(axes,[('input','Generated input'),('baseline','Re-denoise only'),('guided','Previous adaptive')]):
        ax.imshow(Image.open(row[key]));ax.set_title(title);ax.axis('off')
    fig.tight_layout();fig.savefig(Path(row['output'])/'comparison.png',dpi=120);plt.show();plt.close(fig)
SUMMARIES=repair_cases(selected_cases,extractor,checkpoint_sha256=EXPECTED_SHA,revision=REVISION,
    seed=REPAIR_SEED,steps=REPAIR_STEPS,strength=REPAIR_STRENGTH,on_case=show_case,progress=progress)
fig,axes=plt.subplots(len(SUMMARIES),3,figsize=(15,5*len(SUMMARIES)),squeeze=False)
for axes_row,row in zip(axes,SUMMARIES):
    for ax,(key,title) in zip(axes_row,[('input','Generated input'),('baseline','Re-denoise only'),('guided','Previous adaptive')]):
        ax.axis('off');ax.set_title(row['case_id']+'\n'+title)
        if row.get(key):ax.imshow(Image.open(row[key]))
        else:ax.text(.1,.5,'FAILED: '+row.get('error',''),wrap=True)
fig.tight_layout();fig.savefig(OUTPUT/'all_comparisons.png',dpi=100);plt.show();plt.close(fig)
(OUTPUT/'runtime_helpers.json').write_text(json.dumps(RUNTIME_HELPERS,ensure_ascii=False),encoding='utf-8')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
print('모든 성공/실패 결과:',archive)
files.download(archive)